In [ ]:
def main(datasources, start_date, end_date):
    """【1625：合成1399加收盘段季节性与半剂量复活近五日成交笔数根】（= 1619 修正打包：1619 的 notebook 漏了文件末尾的辅助函数，平台报错；代码逻辑与因子值完全不变）= 1399（第十三轮 G4 换手前沿，B1–B13 与 1399 完全相同）＋ B14 收盘段季节性（全天 0.115、14:30 截面再加 0.23；= 第十六轮 v_CLS1399，平台三区间相对 1399 主区间与 2020 五项全好）；A = 1363（九根合成，精简按月、头部按季；与已交付的 1363 逐值相同）（原始值）＋ Σ w × (B 的截面百分位排名 − 0.5)。
      B1 × 0.0614（先做 EWMA α=0.75）：同时段季节性（= 1232 的根，再加权）
      B2 × 0.0139：聪明钱（1236）
      B3 × 0.0073：大单方向（1237）
      B4 × -0.0288：量价同向（1260）
      B5 × 0.0939：零收益分钟占比（1263）
      B6 × 0.0085：近 5 日单笔额离散度（1323）
      B7 × -0.017：近 5 日大单方向（1273）
      B8 × -0.1583：近 60 分钟上涨分钟量占比（1334）
      B9 × 0.0076：近 90 分钟上涨分钟量占比（1342）
      B10 × 0.9：队友 sushirui 的 161
      B11 × 0.0177：日内已实现峰度反向（1381）
      B12 × 0.0275：近 5 日已实现偏度反向（1385）
      B13 × 0.0063：前日收盘竞价价格冲击（1406）
      B14 × 0.115，14:30 截面再加 0.23：收盘段季节性（最后半小时 收盘÷VWAP−1 的过去 20 日均值，收盘集合竞价主导；同一交易日各截面取值相同）
    口径与评测机 values 合成（build_values_composite.py 的 raw 模式逐步叠加）一致：每个 B 在自身全部行上逐截面排名，
    A 有、B 没有的行该项记 0，09:30 置 0，缺失记 0。A 先算完只留结果；每个 B 按自然季度循环计算，控制内存。
    """
    import gc
    import numpy as np
    import pandas as pd

    a = _main_a(datasources, start_date, end_date)[["date", "instrument", "factor"]].copy()
    a["date"] = pd.to_datetime(a["date"])
    a["instrument"] = a["instrument"].astype(str)
    v = pd.to_numeric(a["factor"], errors="coerce").to_numpy(float)
    key = a[["date", "instrument"]]
    _hm = a["date"].dt.strftime("%H%M").to_numpy()
    gc.collect()
    for _fn, _w, _ew, _ex in [
        (main_b1, 0.0614, 0.75, {}),
        (main_b2, 0.0139, 0.0, {}),
        (main_b3, 0.0073, 0.0, {}),
        (main_b4, -0.0288, 0.0, {}),
        (main_b5, 0.0939, 0.0, {}),
        (main_b6, 0.00425, 0.0, {}),
        (main_b7, -0.0085, 0.0, {}),
        (main_b8, -0.1583, 0.0, {}),
        (main_b9, 0.0076, 0.0, {}),
        (main_b10, 0.9, 0.0, {}),
        (main_b11, 0.0177, 0.0, {}),
        (main_b12, 0.0275, 0.0, {}),
        (main_b13, 0.0063, 0.0, {}),
        (main_b14, 0.115, 0.0, {'1430': 0.23}),
    ]:
        parts = []
        for _q in pd.period_range(pd.Timestamp(start_date).to_period("Q"), pd.Timestamp(end_date).to_period("Q"), freq="Q"):
            _s = max(pd.Timestamp(start_date), _q.start_time)
            _e = min(pd.Timestamp(end_date), _q.end_time)
            if _s > _e:
                continue
            parts.append(_fn(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"))[["date", "instrument", "factor"]])
            gc.collect()
        b = pd.concat(parts, ignore_index=True)
        del parts
        b["date"] = pd.to_datetime(b["date"])
        b["instrument"] = b["instrument"].astype(str)
        b = b.drop_duplicates(["date", "instrument"])
        b["factor"] = pd.to_numeric(b["factor"], errors="coerce")
        if _ew > 0:                                                       # 同一股票按时间连续的 EWMA（跨日），只用过去，与 values 合成的 --ewma 相同
            b = b.sort_values(["instrument", "date"]).reset_index(drop=True)
            b["factor"] = b.groupby("instrument", sort=False)["factor"].transform(lambda x: x.ewm(alpha=1 - _ew, adjust=False, ignore_na=True).mean())
        b["rx"] = b.groupby("date")["factor"].rank(pct=True) - 0.5
        rx = key.merge(b[["date", "instrument", "rx"]], on=["date", "instrument"], how="left")["rx"].fillna(0.0).to_numpy(float)
        del b
        v = v + _w * rx
        for _s, _x in _ex.items():                                        # 分时段的额外权重
            v = v + _x * rx * (_hm == _s)
        gc.collect()
    v[a["date"].dt.strftime("%H%M").eq("0930").to_numpy()] = 0.0
    out = pd.DataFrame({"date": a["date"].to_numpy(), "instrument": a["instrument"].to_numpy(), "factor": np.nan_to_num(v, nan=0.0)})
    return out.sort_values(["date", "instrument"]).reset_index(drop=True)


def _main_a(datasources, start_date, end_date):
    """【新根因子 第三轮·多根合成 1399】合成头部加九根含六十九十分钟资金流精简按月头部按季

    B = 490 头部原始值 + Σ w·rank(根)，09:30 置 0，股票池骨架、缺失填 0。根来自多统计量框架 1340_资金流窗口框架.py（原样复用），
    以 L_ 开头的季节性根先在同一股票连续截面上做 EWMA（α=0.75）再排名。各根含义：
      L_lab10  同时段季节性（下一个半小时段过去 10 日"收盘 ÷ VWAP − 1"均值）
      M_q_lin  聪明钱（|r|/v 加权的当日 VWAP ÷ 当日 VWAP − 1）
      N_dir    大单方向（Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|）
      W_vdir   量价同向（Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|）
      W_zero   当日零收益分钟占比
      Y_sd5    近 5 日单笔成交额离散度
      X_n5d    近 5 日大单方向
      Z_mfr60  近 60 分钟上涨分钟成交量占比（资金流）
      Z_mfr90  近 90 分钟上涨分钟成交量占比（资金流）
    权重：{"L_lab10": +0.20, "M_q_lin": +0.10, "N_dir": -0.10, "W_vdir": -0.10, "W_zero": +0.10, "Y_sd5": +0.10, "X_n5d": -0.10, "Z_mfr60": -0.15, "Z_mfr90": -0.10}。口径与评测机 values 合成（build_values_composite.py raw 模式）一致。
    """
    import numpy as np
    import pandas as pd
    import dai

    W = {"L_lab10": +0.20, "M_q_lin": +0.10, "N_dir": -0.10, "W_vdir": -0.10, "W_zero": +0.10, "Y_sd5": +0.05, "X_n5d": -0.05, "Z_mfr60": -0.15, "Z_mfr90": -0.10}
    ALPHA = 0.75
    MONTHLY = True                                                       # True：分钟类根按月分批（控制平台整段调用的内存）
    HEADQ = True                                                           # True：头部按自然季度分批
    _cache = {}
    _roots = {}
    _mparts = {}
    if MONTHLY:                                                               # 分钟类根按月分批：每月一次分钟 SQL，算完该月所有根就释放
        _mins = [st for st in W if not st.startswith("L_")]
        for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
            _s = max(pd.Timestamp(start_date), _ms.start_time)
            _e = min(pd.Timestamp(end_date), _ms.end_time)
            if _s > _e:
                continue
            _c = {}
            for st in _mins:
                _mparts.setdefault(st, []).append(
                    _root_impl(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), st, _c))
            del _c
    for st in W:
        if st.startswith("L_"):
            warm = (pd.to_datetime(start_date) - pd.Timedelta(days=21)).strftime("%Y-%m-%d %H:%M:%S")
            x = _root_impl(datasources, warm, end_date, st, {})
            x["date"] = pd.to_datetime(x["date"])
            x = x.sort_values(["instrument", "date"]).reset_index(drop=True)
            x["factor"] = x.groupby("instrument", sort=False)["factor"].transform(
                lambda s: s.ewm(alpha=1 - ALPHA, adjust=False).mean())
            x = x[x["date"] >= pd.Timestamp(start_date)]
        elif MONTHLY:
            x = pd.concat(_mparts.pop(st), ignore_index=True)
            x["date"] = pd.to_datetime(x["date"])
        else:
            x = _root_impl(datasources, start_date, end_date, st, _cache)
            x["date"] = pd.to_datetime(x["date"])
        x["instrument"] = x["instrument"].astype(str)
        x["r_" + st] = x.groupby("date")["factor"].rank(pct=True) - 0.5
        _roots[st] = x[["date", "instrument", "r_" + st]]
    import gc
    gc.collect()                                                              # 先算完各根（只留很小的结果），最后才算头部：峰值≈头部本身
    if HEADQ:                                                                 # 头部按自然季度循环（与评测机按季度评分逐位一致）
        _hp = []
        for _q in pd.period_range(pd.Timestamp(start_date).to_period("Q"), pd.Timestamp(end_date).to_period("Q"), freq="Q"):
            _s = max(pd.Timestamp(start_date), _q.start_time)
            _e = min(pd.Timestamp(end_date), _q.end_time)
            if _s > _e:
                continue
            _hp.append(_head_impl(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S")))
            gc.collect()
        _head = pd.concat(_hp, ignore_index=True).drop_duplicates(["date", "instrument"])
        del _hp
    else:
        _head = _head_impl(datasources, start_date, end_date)
    _head = _head.rename(columns={"factor": "fh"})
    _head["date"] = pd.to_datetime(_head["date"])
    _head["instrument"] = _head["instrument"].astype(str)
    _m = _head
    for st, x in _roots.items():
        _m = _m.merge(x, on=["date", "instrument"], how="left")
        _m["r_" + st] = _m["r_" + st].fillna(0.0)
    _m["factor"] = _m["fh"].astype("float64")
    for st, w in W.items():
        _m["factor"] = _m["factor"] + w * _m["r_" + st]

    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    _out = _pool.merge(_m[["date", "instrument", "factor"]].drop_duplicates(["date", "instrument"]), on=["date", "instrument"], how="left")
    _out["factor"] = _out["factor"].astype("float64").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    _out.loc[_out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return _out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _head_impl(datasources, start_date, end_date):
    """长窗口软同伴（240 日）+ 六特征 1.5 倍中性化（新增换手率与市值二次项）+（09:30 置零版）。

    148（换手率特征，0.834）与 149（市值二次项，0.834）各自与底座持平，这里同时放进中性化特征集，
      检验两个「官方风格未覆盖的暴露」一起剔除后能否超过 144。
    09:30 修复：同 104 号对照版本。"""
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    # 向前多取数据：同伴相关系数需要月初之前 60 个交易日的日收益（约 90 个自然日，加上月内偏移与长假留足余量）。
    LOOKBACK_DAYS = 400
    PEER_K, PEER_LOOKBACK, PEER_MIN_OBS, PEER_POWER = 200, 240, 120, 2.0
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d %H:%M:%S")

    sql = f"""
    WITH cte_raw AS (
        SELECT
            instrument_id,
            strftime(date, '%Y-%m-%d') AS trading_day,
            strftime(date, '%H%M%S') AS hms,
            CAST(COALESCE(bid_price1, 0) AS DOUBLE) AS bp1,
            CAST(COALESCE(ask_price1, 0) AS DOUBLE) AS ap1,
            CAST(COALESCE(high, 0) AS DOUBLE) AS px_high,
            CAST(COALESCE(low, 0) AS DOUBLE) AS px_low,
            CAST(COALESCE(open, 0) AS DOUBLE) AS px_open,
            CAST(COALESCE(close, 0) AS DOUBLE) AS px_close,
            CAST(COALESCE(pre_close, 0) AS DOUBLE) AS pre_close,
            CAST(COALESCE(volume, 0) AS DOUBLE) AS vol,
            CAST(COALESCE(amount, 0) AS DOUBLE) AS amt,
            CAST(COALESCE(deal_number, 0) AS DOUBLE) AS deals
        FROM {bar1m}
    ),
    cte_snapshot AS (
        SELECT
            instrument_id,
            trading_day,
            hms,
            pre_close,
            px_high,
            px_low,
            deals,
            CASE
                WHEN vol > 0 AND amt > 0 AND px_low > 0
                     AND amt / vol >= px_low * 0.98 AND amt / vol <= px_high * 1.02
                THEN amt ELSE 0
            END AS amt_c,
            CASE
                WHEN vol > 0 AND amt > 0 AND px_low > 0
                     AND amt / vol >= px_low * 0.98 AND amt / vol <= px_high * 1.02
                THEN vol ELSE 0
            END AS vol_c,
            CASE
                WHEN bp1 > 0 AND ap1 > bp1 THEN (bp1 + ap1) / 2.0
                WHEN px_close > 0 THEN px_close
                ELSE 0
            END AS px,
            CASE WHEN px_high > 0 AND px_low > 0 AND px_high >= px_low AND px_open > 0 AND px_close > 0 THEN px_high - px_low ELSE 0 END AS rng_c,
            CASE WHEN px_high > 0 AND px_low > 0 AND px_high >= px_low AND px_open > 0 AND px_close > 0 THEN px_high - GREATEST(px_open, px_close) ELSE 0 END AS upsh_c,
            CASE WHEN px_high > 0 AND px_low > 0 AND px_high >= px_low AND px_open > 0 AND px_close > 0 THEN ABS(px_close - px_open) ELSE 0 END AS body_c,
            CASE
                WHEN hms > '090000' AND hms <= '093000' THEN 93000
                WHEN hms > '093000' AND hms <= '100000' THEN 100000
                WHEN hms > '100000' AND hms <= '103000' THEN 103000
                WHEN hms > '103000' AND hms <= '110000' THEN 110000
                WHEN hms > '110000' AND hms <= '113000' THEN 113000
                WHEN hms > '130000' AND hms <= '133000' THEN 133000
                WHEN hms > '133000' AND hms <= '140000' THEN 140000
                WHEN hms > '140000' AND hms <= '143000' THEN 143000
                WHEN hms > '143000' AND hms <= '150000' THEN 150000
                ELSE -1
            END AS time_segment
        FROM cte_raw
    ),
    cte_window AS (
        SELECT
            trading_day, time_segment, instrument_id,
            SUM(amt_c) AS sum_amt,
            SUM(vol_c) AS sum_vol,
            SUM(body_c) AS sum_body,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment
                      OR (time_segment = 93000 AND hms = '092500') THEN px ELSE 0 END) AS last_px,
            MAX(pre_close) AS pre_close,
            MAX(CASE WHEN px_high > 0 THEN px_high END) AS win_high,
            MIN(CASE WHEN px_low > 0 THEN px_low END) AS win_low,
            SUM(rng_c) AS sum_rng,
            SUM(upsh_c) AS sum_upsh,
            SUM(deals) AS sum_deals
        FROM cte_snapshot
        WHERE time_segment != -1
        GROUP BY instrument_id, trading_day, time_segment
    )
    SELECT
        CAST(CONCAT(
            f.trading_day,
            ' ',
            strftime(strptime(LPAD(f.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        f.sum_amt, f.sum_vol, f.sum_body, f.last_px, f.pre_close, f.win_high, f.win_low, f.sum_rng, f.sum_upsh, f.sum_deals
    FROM cte_window f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={"date": [query_start, end_date]}, compression=True).df()
    df = df.dropna(subset=["instrument"]).drop_duplicates(["date", "instrument"])
    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    for c in ["sum_amt", "sum_vol", "sum_body", "last_px", "pre_close", "win_high", "win_low", "sum_rng", "sum_upsh", "sum_deals"]:
        df[c] = df[c].astype("float64")
    df["last_px"] = df["last_px"].where(df["last_px"] > 0)
    df["pre_close"] = df["pre_close"].where(df["pre_close"] > 0)

    # 股票池：同伴只在股票池内挑选（向前取历史区间的成分，取不到时用评估区间的成分）。
    instruments_table = datasources.get("instruments", "cpt_jyc_2026_instruments")
    exposure_table = datasources.get("exposure", "cpt_jyc_2026_exposure")
    pool_hist = dai.query(
        f"SELECT date, instrument FROM {instruments_table}",
        filters={"date": [query_start, end_date]},
        compression=True,
    ).df()
    pool_hist["date"] = pd.to_datetime(pool_hist["date"])
    pool_hist["instrument"] = pool_hist["instrument"].astype(str)
    stk_pool = pool_hist[(pool_hist["date"] >= pd.to_datetime(start_date)) & (pool_hist["date"] <= pd.to_datetime(end_date))]
    universe = set(pool_hist["instrument"].unique())
    df = df[df["instrument"].isin(universe)]
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
    df["day"] = df["date"].dt.normalize()

    # 官方暴露表（日频）里的流通市值：用于构造「官方风格没有覆盖」的中性化特征。
    _exp = dai.query(
        "SELECT date, instrument, float_market_cap FROM %s" % exposure_table,
        filters={"date": [query_start, end_date]}, compression=True,
    ).df()
    _exp["day"] = pd.to_datetime(_exp["date"]).dt.normalize()
    _exp["instrument"] = _exp["instrument"].astype(str)
    _mcap = _exp.groupby(["instrument", "day"], sort=False)["float_market_cap"].last()
    # 暴露表第 t 天的流通市值按当天 15:00 收盘价计算，盘中截面不可得：一律取该股票前一交易日的值。
    _mcap = _mcap.sort_index().groupby(level=0).shift(1)
    _midx = pd.MultiIndex.from_arrays([df["instrument"].values, df["day"].values])
    df["mcap"] = pd.Series(_mcap.reindex(_midx).to_numpy(), index=df.index)
    df["mcap"] = df["mcap"].where(df["mcap"] > 0)
    # 官方 10 风格暴露 + 行业码（与平台标签残差化口径一致），用于因子自身的中性化测试。
    _st = dai.query(
        "SELECT date, instrument, SIZE, BETA, MOMENTUM, RESVOL, SIZENL, BTOP, LIQUIDTY, EARNYILD, GROWTH, LEVERAGE, industry_level1_code FROM %s" % exposure_table,
        filters={"date": [query_start, end_date]}, compression=True,
    ).df()
    _st["day"] = pd.to_datetime(_st["date"]).dt.normalize()
    _st["instrument"] = _st["instrument"].astype(str)
    _st = _st.drop_duplicates(["instrument", "day"])
    # 风格暴露与行业同样取前一交易日的值（第 t 天的暴露在当天收盘后才可得）。
    _st = _st.sort_values(["instrument", "day"]).reset_index(drop=True)
    for _c in ["SIZE", "BETA", "MOMENTUM", "RESVOL", "SIZENL", "BTOP", "LIQUIDTY", "EARNYILD", "GROWTH", "LEVERAGE", "industry_level1_code"]:
        _st[_c] = _st.groupby("instrument")[_c].shift(1)
    _st = _st.dropna(subset=["industry_level1_code"])
    _sg = _st.set_index(["instrument", "day"])
    _sid = pd.MultiIndex.from_arrays([df["instrument"].values, df["day"].values])
    for _c in ['SIZE', 'BETA', 'MOMENTUM', 'RESVOL', 'SIZENL', 'BTOP', 'LIQUIDTY', 'EARNYILD', 'GROWTH', 'LEVERAGE']:
        df["s_" + _c] = pd.Series(_sg[_c].reindex(_sid).to_numpy(), index=df.index)
    _ind = _sg["industry_level1_code"].astype(str)
    for _c in sorted(_ind.dropna().unique()):
        df["i_" + _c] = (_ind.reindex(_sid).to_numpy() == _c).astype(float)

    def safe_log_ratio(num, den):
        ok = (num > 0) & (den > 0)
        return pd.Series(np.where(ok, np.log(num.where(ok, 1.0) / den.where(ok, 1.0)), np.nan), index=num.index)

    def vwap_of(amt, vol):
        return (amt / vol.where(vol > 0)).where(amt > 0)

    def roll(s, k, fn, min_periods):
        return s.groupby(df["instrument"], sort=False).transform(lambda x: getattr(x.rolling(k, min_periods=min_periods), fn)())

    grp = df.groupby("instrument", sort=False)
    grp_day = df.groupby(["instrument", "day"], sort=False)
    same_day = grp["day"].shift(1) == df["day"]
    px = df["last_px"]

    day_vwap = vwap_of(grp_day["sum_amt"].cumsum(), grp_day["sum_vol"].cumsum())
    r_dvwap = safe_log_ratio(day_vwap, px)

    def rolling_vwap_rev(k):
        return safe_log_ratio(vwap_of(roll(df["sum_amt"], k, "sum", 1), roll(df["sum_vol"], k, "sum", 1)), px).fillna(r_dvwap)

    df["r_30"] = safe_log_ratio(grp["last_px"].shift(1).where(same_day, df["pre_close"]), px)
    df["r_rv4"] = rolling_vwap_rev(4)
    df["r_rv8"] = rolling_vwap_rev(8)
    df["r_rv16"] = rolling_vwap_rev(16)
    df["r_day"] = safe_log_ratio(df["pre_close"], px)

    # 个股 beta（40 窗口滚动回归，截断 [-1, 4]，不足取 1），用于特质波动。
    win_ret = safe_log_ratio(px, grp["last_px"].shift(1))
    ret30 = safe_log_ratio(px, grp["last_px"].shift(1).where(same_day)).fillna(safe_log_ratio(px, df["pre_close"]))
    mkt30 = ret30.groupby(df["date"]).transform("mean")
    ex, ey = roll(ret30, 40, "mean", 10), roll(mkt30, 40, "mean", 10)
    exy, ey2 = roll(ret30 * mkt30, 40, "mean", 10), roll(mkt30 ** 2, 40, "mean", 10)
    var_m = (ey2 - ey ** 2).where((ey2 - ey ** 2) > 1e-12)
    beta = ((exy - ex * ey) / var_m).clip(-1, 4).fillna(1.0)

    # 股票特征：规模（-ln 最近 40 窗口成交额）、特质波动（取负）、非流动性（Amihud）、低价、MAX（取负）。
    df["c_size"] = -np.log1p(roll(df["sum_amt"], 40, "sum", 10))
    idio = win_ret - beta * win_ret.groupby(df["date"]).transform("mean")
    df["c_ivol"] = -roll(idio, 40, "std", 10)
    df["c_illiq"] = roll(win_ret.abs() / (df["sum_amt"] / 1e8).where(df["sum_amt"] > 0), 40, "mean", 10)
    df["c_lowpx"] = -np.log(px.where(px > 0))

    # 换手率代理：最近 40 个窗口成交额 ÷ 流通市值（官方风格里没有直接的换手率暴露）。
    df["c_turn"] = roll(df["sum_amt"], 40, "sum", 10) / df["mcap"]

    # 流通市值的非线性项：ln(流通市值) 的平方（官方 SIZE 风格只覆盖线性部分）。
    df["c_mcap2"] = np.log(df["mcap"]) ** 2
    df["c_max"] = -roll(win_ret, 40, "max", 10)
    # 分钟上影线占比：窗口内 Σ(最高-实体上沿)/Σ(最高-最低)，40 窗口均值取负（冲高回落越多，反转越强）。
    _upsh = df["sum_upsh"] / df["sum_rng"].where(df["sum_rng"] > 0)
    df["c_body"] = -roll(df["sum_body"] / df["sum_rng"].where(df["sum_rng"] > 0), 40, "mean", 10)
    df["c_upsh"] = -_upsh.groupby(df["instrument"], sort=False).transform(lambda x: x.ewm(span=10, min_periods=8).mean())
    # 平均单笔成交额：Σ成交额/Σ委托笔数 的 40 窗口均值（大单/小单主导窗口的代理）。
    _avgtr = df["sum_amt"] / df["sum_deals"].where(df["sum_deals"] > 0)
    df["c_avgtr"] = roll(_avgtr, 40, "mean", 10)

    # 同伴：每个自然月月初，用此前 PEER_LOOKBACK 个交易日的日收益（收盘段价格 / 昨收）计算相关系数。
    daily = df.groupby(["day", "instrument"], sort=False).agg(px_last=("last_px", "last"), pc=("pre_close", "first"))
    daily_ret = safe_log_ratio(daily["px_last"], daily["pc"]).unstack("instrument").sort_index()
    peer_insts = daily_ret.columns.values
    daily_amt = df.groupby(["day", "instrument"], sort=False)["sum_amt"].sum().unstack("instrument").reindex(columns=peer_insts).sort_index()
    all_days = daily_ret.index
    day_month = all_days.to_period("M")
    peers_by_month = {}
    out_months = pd.to_datetime(stk_pool["date"]).dt.normalize().dt.to_period("M").unique()
    for m in out_months:
        in_month = all_days[day_month == m]
        if len(in_month) == 0:
            continue
        hist = daily_ret.loc[daily_ret.index < in_month[0]].tail(PEER_LOOKBACK)
        if len(hist) < PEER_MIN_OBS:
            continue
        ok = (hist.notna().sum() >= PEER_MIN_OBS).values
        corr = hist.loc[:, ok].corr(min_periods=PEER_MIN_OBS).values
        np.fill_diagonal(corr, np.nan)
        corr = np.nan_to_num(corr, nan=-1.0)          # 缺失相关性记 -1（等价于不选）
        sub = np.where(ok)[0]
        top = np.argsort(-corr, axis=1, kind="stable")[:, :PEER_K]
        idx = np.full((len(peer_insts), PEER_K), -1, dtype=np.int64)
        idx[sub] = sub[top]
        cvals = corr[np.arange(len(corr))[:, None], top]
        # 规模加权：相关度^p × 该同伴前 PEER_LOOKBACK 日的平均成交额（大票的共同运动信息权重更高）。
        _aw = daily_amt.loc[daily_amt.index < in_month[0]].tail(PEER_LOOKBACK).mean().fillna(0.0).values
        _aw = np.where(_aw > 0, _aw, 0.0)
        wts = (np.clip(cvals, 0.0, None) ** PEER_POWER) * _aw[sub][:, None]
        wsum = wts.sum(axis=1, keepdims=True)
        wts = np.where(wsum > 0, wts / np.where(wsum > 0, wsum, 1.0), 0.0)
        full_w = np.zeros((len(peer_insts), PEER_K))
        full_w[sub] = wts
        peers_by_month[m] = (idx, full_w)

    df = df.replace([np.inf, -np.inf], np.nan)
    keep = ["r_30", "r_rv4", "r_rv8", "r_day", "c_size", "c_ivol", "c_illiq", "c_lowpx", "c_max", "c_turn", "c_mcap2", "c_upsh", "c_avgtr", "r_rv16", "c_body"]
    keep = keep + sorted([c for c in df.columns if c.startswith("i_")])
    result = pd.merge(df[["date", "instrument"] + keep], stk_pool[["date", "instrument"]], how="inner", on=["date", "instrument"])
    result = result.drop_duplicates(["date", "instrument"]).sort_values(["date", "instrument"]).reset_index(drop=True)
    result_month = result["date"].dt.to_period("M")

    def peer_mean(col):
        """每个截面上同伴的 col 均值（至少 3 个有效值），没有同伴信息时为缺失。"""
        out = np.full(len(result), np.nan)
        for m, rows in result.groupby(result_month, sort=False).indices.items():
            rec = peers_by_month.get(m)
            if rec is None:
                continue
            idx, wts = rec
            sub = result.iloc[rows]
            wide = sub.pivot(index="date", columns="instrument", values=col).reindex(columns=peer_insts)
            valid = idx >= 0
            g = wide.values[:, np.where(valid, idx, 0)]
            g = np.where(valid[None, :, :], g, np.nan)
            ww = np.where(valid[None, :, :], wts[None, :, :], 0.0)
            okm = ~np.isnan(g)
            cnt = okm.sum(axis=2)
            wsum = (ww * okm).sum(axis=2)
            pm = np.where((cnt >= 3) & (wsum > 0), np.nansum(np.where(okm, g, 0.0) * ww, axis=2) / np.where(wsum > 0, wsum, 1.0), np.nan)
            r_pos = wide.index.get_indexer(sub["date"])
            c_pos = pd.Index(peer_insts).get_indexer(sub["instrument"])
            out[rows] = np.where(c_pos >= 0, pm[r_pos, np.maximum(c_pos, 0)], np.nan)
        return pd.Series(out, index=result.index)

    def rank_blend(cols, weights=None):
        """各分量在股票池内逐截面居中排名后加权平均（缺失分量不参与，权重按可用分量重新归一）。"""
        weights = weights or [1.0] * len(cols)
        ranks = (result.groupby("date")[cols].rank(pct=True) - 0.5).to_numpy()
        w = np.array(weights, dtype=float)
        has = ~np.isnan(ranks)
        den = (has * w).sum(axis=1)
        num = np.nansum(np.nan_to_num(ranks) * w, axis=1)
        return pd.Series(np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan), index=result.index)

    def neutralize(col, chars, frac=1.0):
        """截面中性化：col 的居中排名对各特征居中排名（缺失记 0）逐截面 OLS（含截距），返回 排名 - frac x 拟合值。"""
        y = result.groupby("date")[col].rank(pct=True) - 0.5
        X = np.column_stack([(result.groupby("date")[c].rank(pct=True) - 0.5).fillna(0.0).values for c in chars])
        ok = y.notna().values
        codes, uniq = pd.factorize(result["date"].values)
        k, n_grp = X.shape[1], len(uniq)
        Xc = np.column_stack([np.ones(len(result)), X]) * ok[:, None]
        yv = np.nan_to_num(y.values) * ok
        xtx = np.zeros((n_grp, k + 1, k + 1))
        xty = np.zeros((n_grp, k + 1))
        for i in range(k + 1):
            xty[:, i] = np.bincount(codes, Xc[:, i] * yv, minlength=n_grp)
            for j in range(i, k + 1):
                v = np.bincount(codes, Xc[:, i] * Xc[:, j], minlength=n_grp)
                xtx[:, i, j] = v
                xtx[:, j, i] = v
        b = np.linalg.solve(xtx + 1e-9 * np.eye(k + 1), xty[..., None])[..., 0]
        fit = (X * b[codes, 1:]).sum(axis=1)
        return pd.Series(np.where(ok, y.values - frac * fit, np.nan), index=result.index)

    def peer_ladder():
        cols = []
        for c in ["r_30", "r_rv4", "r_rv8", "r_day"]:
            result[c + "_p"] = result[c] - peer_mean(c).fillna(result.groupby("date")[c].transform("mean"))
            cols.append(c + "_p")
        return rank_blend(cols)

    CHARS4 = ["c_size", "c_ivol", "c_illiq", "c_lowpx", "c_turn", "c_mcap2"] + sorted([c for c in df.columns if c.startswith("i_")])
    result["ladder"] = peer_ladder()
    result["signal"] = rank_blend(["ladder", "c_upsh", "c_avgtr", "r_rv16", "c_body"], [1.0, 0.30, 0.15, 0.20, 0.15])
    result["factor"] = neutralize("signal", CHARS4, 2.5)
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan).astype("float64").fillna(0.0)
    # 09:30 输出 0（与 104/103/107 同口径）：平台跳过全 0 截面。
    result.loc[result["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return result[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"]
    ).reset_index(drop=True)


def _root_impl(datasources, start_date, end_date, STAT, _cache):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *, ABS(r) / GREATEST(v, 1) AS s2,
                AVG(lats) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS lats_prev,
                AVG(lv) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS lv_prev
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv, SUM(s2 * a) OVER w AS cs2a, SUM(s2 * v) OVER w AS cs2v,
                SUM(r * (lats - lats_prev)) OVER w AS cnd,
                SUM(ABS(r)) OVER w AS cabs,
                SUM(r * (lv - lv_prev)) OVER w AS cvd, SUM(CASE WHEN r = 0 THEN 1 ELSE 0 END) OVER w AS czero, COUNT(r) OVER w AS cnr, SUM(lats) OVER w AS sl, SUM(lats * lats) OVER w AS sl2, COUNT(lats) OVER w AS nl,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w60 AS vup60, SUM(v) OVER w60 AS v60,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w90 AS vup90, SUM(v) OVER w90 AS v90
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430', '1500')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        if "min" not in _cache:                                                    # 同一调用内只查一次
            _cache["min"] = _w5_safe(dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df())
        df = _cache["min"].copy()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "M_q_lin": lambda: (df["cs2a"] / df["cs2v"].where(df["cs2v"] > 0)) / vw - 1,
                "N_dir": lambda: df["cnd"] / cabs,
                "W_vdir": lambda: df["cvd"] / cabs,
                "X_n5d": lambda: df["nd5"] / df["a5d"].where(df["a5d"] > 0),
                "Y_sd5": lambda: df["sd5"],
                "Z_mfr60": lambda: df["vup60"] / df["v60"].where(df["v60"] > 0),
                "Z_mfr90": lambda: df["vup90"] / df["v90"].where(df["v90"] > 0),
                "W_zero": lambda: df["czero"] / df["cnr"].where(df["cnr"] > 0),
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b1(datasources, start_date, end_date):
    """【1398：临时L_lab10】新根单独版（公榜探针），STAT = L_lab10，方向 正向。
    根 = 1340_资金流窗口框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b1(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "L_lab10"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b1(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *,
                ABS(r) / SQRT(GREATEST(v, 1)) AS s1, ABS(r) / GREATEST(v, 1) AS s2, ABS(r) / POW(GREATEST(v, 1), 0.75) AS s3, ABS(r) / GREATEST(a, 1) AS s4,
                CASE WHEN bo > 0 AND ao > 0 AND bv > 0 AND av > 0 THEN LN((bv / bo) / (av / ao)) END AS osz,
                LN((bv1 + 1) / (av1 + 1)) AS q1,
                CASE WHEN ap1 > 0 AND bp1 > 0 AND ap1 >= bp1 THEN (ap1 - bp1) / ((ap1 + bp1) / 2) END AS spr,
                r * LAG(r) OVER (PARTITION BY instrument_id, d ORDER BY date) AS rr1,
                AVG(r * r) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS rv_prev,
                COUNT(r) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS n_prev,
                AVG(lats) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS lats_prev,
                AVG(lv) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS lv_prev,
                CASE WHEN bo + ao > 0 THEN (bo - ao) / (bo + ao) END AS cimb,
                CASE WHEN h >= l AND c > 0 THEN (h - l) / c END AS hlr
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(s1 * a) OVER w AS cs1a, SUM(s1 * v) OVER w AS cs1v, SUM(s2 * a) OVER w AS cs2a, SUM(s2 * v) OVER w AS cs2v,
                SUM(r * (lats - lats_prev)) OVER w AS cnd,
                SUM(CASE WHEN lats > lats_prev THEN r ELSE 0 END) OVER w AS cbig,
                SUM(CASE WHEN lats <= lats_prev THEN r ELSE 0 END) OVER w AS csmall,
                SUM(ABS(r)) OVER w AS cabs, SUM(r) OVER w AS csum, SUM(r * r) OVER w AS crr, SUM(rr1) OVER w AS crr1,
                SUM(CASE WHEN n_prev >= 10 AND ABS(r) > 4 * SQRT(rv_prev) THEN 1 ELSE 0 END) OVER w AS cnj,
                SUM(CASE WHEN n_prev >= 10 AND ABS(r) > 4 * SQRT(rv_prev) THEN r ELSE 0 END) OVER w AS csj,
                AVG(osz) OVER w AS mosz,
                AVG(osz) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW) AS mosz30,
                AVG(q1) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS mq5,
                AVG(spr) OVER w AS mspr,
                SUM(r * (lv - lv_prev)) OVER w AS cvd, AVG(cimb) OVER w AS mcimb,
                SUM(v * v) OVER w AS cvv, SUM(CASE WHEN r = 0 THEN 1 ELSE 0 END) OVER w AS czero, COUNT(r) OVER w AS cnr,
                STDDEV_SAMP(lats) OVER w AS sdlats, SUM(hlr) OVER w AS chl,
                SUM(s3 * a) OVER w AS cs3a, SUM(s3 * v) OVER w AS cs3v, SUM(s4 * a) OVER w AS cs4a, SUM(s4 * v) OVER w AS cs4v,
                SUM(r) OVER w60 AS r60, SUM(ABS(r)) OVER w60 AS a60, SUM(r) OVER w30 AS r30, SUM(ABS(r)) OVER w30 AS a30,
                SUM(r) OVER w5 AS r5d, SUM(ABS(r)) OVER w5 AS a5d, SUM(r * (lats - lats_prev)) OVER w5 AS nd5,
                SUM(a) OVER w5 AS a_5, SUM(v) OVER w5 AS v_5, SUM(s2 * a) OVER w5 AS s2a_5, SUM(s2 * v) OVER w5 AS s2v_5,
                SUM(r * (lv - lv_prev)) OVER w5 AS vd5, SUM(CASE WHEN r = 0 THEN 1 ELSE 0 END) OVER w5 AS z5,
                COUNT(r) OVER w5 AS n5, STDDEV_SAMP(lats) OVER w5 AS sd5,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w AS vup, SUM(v) OVER w AS vall,
                SUM(CASE WHEN c >= ap1 THEN a WHEN c <= bp1 THEN -a ELSE 0 END) OVER w AS abid, SUM(a) OVER w AS aall,
                SUM(CASE WHEN r > 0 THEN a ELSE 0 END) OVER w AS aup, SUM(CASE WHEN r > 0 THEN n ELSE 0 END) OVER w AS nup,
                SUM(CASE WHEN r < 0 THEN a ELSE 0 END) OVER w AS adn, SUM(CASE WHEN r < 0 THEN n ELSE 0 END) OVER w AS ndn,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w60 AS vup60, SUM(v) OVER w60 AS v60,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w30 AS vup30, SUM(v) OVER w30 AS v30,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w90 AS vup90, SUM(v) OVER w90 AS v90,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w120 AS vup120, SUM(v) OVER w120 AS v120,
                SUM(CASE WHEN r < 0 THEN v ELSE 0 END) OVER w60 AS vdn60,
                SUM(CASE WHEN r > 0 THEN a ELSE 0 END) OVER w60 AS aup60, SUM(a) OVER w60 AS am60,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w60x AS vup60x, SUM(v) OVER w60x AS v60x
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "M_q": (df["cs1a"] / df["cs1v"].where(df["cs1v"] > 0)) / vw - 1,
                "M_q_lin": (df["cs2a"] / df["cs2v"].where(df["cs2v"] > 0)) / vw - 1,
                "N_dir": df["cnd"] / cabs,
                "N_big": df["cbig"] - df["csmall"],
                "Q_eff": df["csum"].abs() / cabs,
                "Q_seff": df["csum"] / cabs,
                "Q_ac1": df["crr1"] / df["crr"].where(df["crr"] > 0),
                "U_nj": df["cnj"].astype(float),
                "U_sj": df["csj"],
                "S_osz": df["mosz"],
                "S_osz30": df["mosz30"],
                "O_q5": df["mq5"],
                "R_round": (df["c"] - np.round(df["c"])) / df["c"],
                "R_half": (df["c"] - np.round(df["c"] * 2) / 2) / df["c"],
                "V_spr": df["mspr"],
                "W_vdir": df["cvd"] / cabs,
                "X_q60": df["r60"] / df["a60"].where(df["a60"] > 0),
                "X_q30": df["r30"] / df["a30"].where(df["a30"] > 0),
                "X_q5d": df["r5d"] / df["a5d"].where(df["a5d"] > 0),
                "X_n5d": df["nd5"] / df["a5d"].where(df["a5d"] > 0),
                "X_m5d": (df["s2a_5"] / df["s2v_5"].where(df["s2v_5"] > 0)) / (df["a_5"] / df["v_5"].where(df["v_5"] > 0)) - 1,
                "X_m75": (df["cs3a"] / df["cs3v"].where(df["cs3v"] > 0)) / vw - 1,
                "X_ma": (df["cs4a"] / df["cs4v"].where(df["cs4v"] > 0)) / vw - 1,
                "Y_vd5": df["vd5"] / df["a5d"].where(df["a5d"] > 0),
                "Y_z5": df["z5"] / df["n5"].where(df["n5"] > 0),
                "Y_sd5": df["sd5"],
                "Y_mfr": df["vup"] / df["vall"].where(df["vall"] > 0),
                "Y_bid": df["abid"] / df["aall"].where(df["aall"] > 0),
                "Z_aup": df["aup"] / df["aall"].where(df["aall"] > 0),
                "Z_mfr60": df["vup60"] / df["v60"].where(df["v60"] > 0),
                "Z_ats": np.log((df["aup"] / df["nup"].where(df["nup"] > 0)) / (df["adn"] / df["ndn"].where(df["ndn"] > 0))),
                "Z_mfr30": df["vup30"] / df["v30"].where(df["v30"] > 0),
                "Z_mfr90": df["vup90"] / df["v90"].where(df["v90"] > 0),
                "Z_mfr120": df["vup120"] / df["v120"].where(df["v120"] > 0),
                "Z_net60": (df["vup60"] - df["vdn60"]) / df["v60"].where(df["v60"] > 0),
                "Z_amfr60": df["aup60"] / df["am60"].where(df["am60"] > 0),
                "Z_mfr60x": df["vup60x"] / df["v60x"].where(df["v60x"] > 0),
                "W_cnt": df["mcimb"],
                "W_hhi": df["cvv"] / (df["cv"] ** 2).where(df["cv"] > 0),
                "W_zero": df["czero"] / df["cnr"].where(df["cnr"] > 0),
                "W_atsd": df["sdlats"],
                "W_hl": df["chl"] / df["csum"].abs().where(df["csum"].abs() > 0),
            }[STAT.split("_res")[0]]
        df["signal"] = np.asarray(sig, dtype=float)
        if "_res" in STAT:                                                   # 逐截面对当日收益（_res2 再加量价同向）回归取残差
            Xr = [df["csum"].to_numpy(float)]
            if STAT.endswith("_res2"):
                Xr.append((df["cvd"] / cabs).to_numpy(float))
            Xr = np.column_stack(Xr)
            y_ = df["signal"].to_numpy(float)
            out_ = np.full(len(df), np.nan)
            for _, ii in df.groupby("date").indices.items():
                ok_ = np.isfinite(y_[ii]) & np.isfinite(Xr[ii]).all(1)
                if ok_.sum() < 50:
                    continue
                A_ = np.column_stack([np.ones(ok_.sum()), Xr[ii][ok_]])
                coef_, *_ = np.linalg.lstsq(A_, y_[ii][ok_], rcond=None)
                out_[ii[ok_]] = y_[ii][ok_] - A_ @ coef_
            df["signal"] = out_
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b2(datasources, start_date, end_date):
    """【1398：临时M_q_lin】新根单独版（公榜探针），STAT = M_q_lin，方向 正向。
    根 = 1340_资金流窗口框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b2(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "M_q_lin"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b2(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *, ABS(r) / GREATEST(v, 1) AS s2
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv, SUM(s2 * a) OVER w AS cs2a, SUM(s2 * v) OVER w AS cs2v,
                SUM(ABS(r)) OVER w AS cabs
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "M_q_lin": lambda: (df["cs2a"] / df["cs2v"].where(df["cs2v"] > 0)) / vw - 1,
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b3(datasources, start_date, end_date):
    """【1398：临时N_dir】新根单独版（公榜探针），STAT = N_dir，方向 正向。
    根 = 1340_资金流窗口框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b3(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "N_dir"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b3(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *,
                AVG(lats) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS lats_prev
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(r * (lats - lats_prev)) OVER w AS cnd,
                SUM(ABS(r)) OVER w AS cabs
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "N_dir": lambda: df["cnd"] / cabs,
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b4(datasources, start_date, end_date):
    """【1398：临时W_vdir】新根单独版（公榜探针），STAT = W_vdir，方向 正向。
    根 = 1340_资金流窗口框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b4(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "W_vdir"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b4(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *,
                AVG(lv) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS lv_prev
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(ABS(r)) OVER w AS cabs,
                SUM(r * (lv - lv_prev)) OVER w AS cvd
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "W_vdir": lambda: df["cvd"] / cabs,
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b5(datasources, start_date, end_date):
    """【1398：临时W_zero】新根单独版（公榜探针），STAT = W_zero，方向 正向。
    根 = 1340_资金流窗口框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b5(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "W_zero"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b5(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(ABS(r)) OVER w AS cabs, SUM(CASE WHEN r = 0 THEN 1 ELSE 0 END) OVER w AS czero, COUNT(r) OVER w AS cnr
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "W_zero": lambda: df["czero"] / df["cnr"].where(df["cnr"] > 0),
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b6(datasources, start_date, end_date):
    """【1398：临时Y_sd5】新根单独版（公榜探针），STAT = Y_sd5，方向 正向。
    根 = 1340_资金流窗口框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b6(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "Y_sd5"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b6(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(ABS(r)) OVER w AS cabs, SUM(lats) OVER w AS sl, SUM(lats * lats) OVER w AS sl2, COUNT(lats) OVER w AS nl
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430', '1500')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = _w5_safe(dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df())
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "Y_sd5": lambda: df["sd5"],
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b7(datasources, start_date, end_date):
    """【1398：临时X_n5d】新根单独版（公榜探针），STAT = X_n5d，方向 正向。
    根 = 1340_资金流窗口框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b7(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "X_n5d"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b7(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *,
                AVG(lats) OVER (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING) AS lats_prev
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(ABS(r)) OVER w AS cabs, SUM(r * (lats - lats_prev)) OVER w AS cnd
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430', '1500')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = _w5_safe(dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df())
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "X_n5d": lambda: df["nd5"] / df["a5d"].where(df["a5d"] > 0),
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b8(datasources, start_date, end_date):
    """【1398：临时Z_mfr60】新根单独版（公榜探针），STAT = Z_mfr60，方向 正向。
    根 = 1340_资金流窗口框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b8(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "Z_mfr60"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b8(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(ABS(r)) OVER w AS cabs,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w60 AS vup60, SUM(v) OVER w60 AS v60
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "Z_mfr60": lambda: df["vup60"] / df["v60"].where(df["v60"] > 0),
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b9(datasources, start_date, end_date):
    """【1398：临时Z_mfr90】新根单独版（公榜探针），STAT = Z_mfr90，方向 正向。
    根 = 1340_资金流窗口框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b9(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "Z_mfr90"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b9(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(ABS(r)) OVER w AS cabs,
                SUM(CASE WHEN r > 0 THEN v ELSE 0 END) OVER w90 AS vup90, SUM(v) OVER w90 AS v90
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "Z_mfr90": lambda: df["vup90"] / df["v90"].where(df["v90"] > 0),
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b10(datasources, start_date, end_date):
    """
    152（倍增阶梯 + 四类价格锚点四网络 IQR + 低波 / 小盘 / 低价）的 IQR 内加 5 个盘口 / 价格锚点分量。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    161 在 152 号基础上只在 IQR 内部加以下反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_qmid : ln(最近 8 个窗口末一档中间价均值 / 当前一档中间价)，涨跌停单边盘口的窗口不参与锚点、当期缺失
      r_cmid : 最近 40 个窗口 ln(窗口末当前价 / 窗口末成交价) 的均值（成交持续落在买一侧即卖压时为负）
      r_deep : ln(最近 8 个窗口末价按三档挂单总量加权的均价 / 当前价)，挂单厚的窗口价格更可信
      r_micro : ln(最近 8 个窗口成交 VWAP / 当前一档微价格)，微价格按对侧挂单量加权买一 / 卖一价
      r_dealpx : ln(最近 8 个窗口末价按窗口成交笔数加权的均价 / 当前价)
    155 的四个盘口锚点分量再加 deal_px8，来自 155 组合上的逐个加分量筛选（验证段）。
    窗口长度沿用已有的 8 与 40，无新增参数，所有候选统一用同一套网络与聚合。
    没有当前价（09:30 截面）时新分量都缺失，该截面与 152 一样整体补 0。15 分量 x 四网络 = 60 列做 IQR Winsor 均值。
    以下为 152 的说明。
    更长倍增阶梯 + 价格、区间中点与最低价中位数偏离 + 低振幅加权价格锚点 十分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    152 在 151 号基础上去掉两个分量 r_lo（当日低点距离）与 r_slope（VWAP 斜率），其余（同伴网络、中性化、聚合、慢特征融合）全部不变，
    即 146 的 6 个阶梯分量 + r_med / r_mid / r_lomed / r_calm 四个价格锚点，10 分量 x 四网络 = 40 列做 IQR Winsor 均值。
    原因：窗口末价格取截面那一分钟，09:30 截面没有这一分钟（1 分钟数据只有 09:25 竞价和 09:31 之后），当前价缺失。
    146 在该截面所有分量缺失、整体补 0 而不参与评估；r_slope 只用成交额与成交量、不含当前价，147–151 在该截面只剩它一个分量，
    该截面 IC 仅约 0.02，拖低整体（本地按同一口径复现：147 相对 146 估计分差 -0.13，与平台 -0.11 一致）。
    152 的所有分量都以当前价为分母，09:30 截面与 146 相同地整体补 0。四个锚点作为一组加入 146 的 IQR，
    在 09:30 截面按平台口径评估时验证段与保留段同时为正；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.3892 / 0.3850（146 0.4346 / 0.4256）；
      IC 0.08417 / 0.06863（146 0.08398 / 0.06917）；
      ICIR 1.7965 / 1.1614（146 1.7811 / 1.1693）；
      Sharpe 53.06 / 35.48（146 53.27 / 35.88）；
      压力 1.7921 / 1.1388（146 1.7593 / 1.1520）。
    以下为 151 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格、区间中点与最低价中位数偏离 + 低振幅加权价格锚点 十二分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    151 在 150 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_calm : 低振幅加权价格锚点偏离 ln(锚点 / 当前价)，锚点为最近 8 个窗口末价按 1 / (ln(最高 / 最低) + 0.001) 加权的均值。
    价格稳定（振幅小）的窗口里成交价更接近共识价值，权重更大；剧烈波动窗口的价格噪声大，权重更小。
    窗口长度沿用已有的 8，0.001 只是防止零振幅除零的下限，无新增可调参数。12 分量 x 四网络 = 48 列做 IQR Winsor 均值。
    这一分量来自第十七批“换权重口径锚点”信号的分量级筛选（逐个作为新分量放进 150 的 IQR），
    验证段与保留段同时为正且验证段增量最高；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4174 / 0.4117（150 0.4267 / 0.4194）；
      IC 0.07968 / 0.06546（150 0.08003 / 0.06574）；
      ICIR 1.6943 / 1.1368（150 1.7005 / 1.1415）；
      Sharpe 47.72 / 34.28（150 47.56 / 34.33）；
      压力 1.6870 / 1.1431（150 1.6900 / 1.1467）。
    以下为 150 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格、区间中点与最低价中位数偏离 十一分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    150 在 149 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_lomed : 最低价中位数偏离 ln(最近 8 个窗口最低价的中位数 / 当前价)。
    r_lo 只看当日累计最低点这一个极值，r_lomed 取近 8 个窗口各自低点的中位数，是稳健的近期支撑位；
    窗口长度沿用已有的 8，无新增参数。11 分量 x 四网络 = 44 列做 IQR Winsor 均值。
    这一分量来自第十四至十八批“锚点”信号的分量级筛选（逐个作为新分量放进 149 的 IQR，共 20 个），
    验证段与保留段同时为正且两段增量最高；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4267 / 0.4194（149 0.4303 / 0.4236）；
      IC 0.08003 / 0.06574（149 0.07995 / 0.06575）；
      ICIR 1.7005 / 1.1415（149 1.6975 / 1.1403）；
      Sharpe 47.56 / 34.33（149 47.31 / 34.16）；
      压力 1.6900 / 1.1467（149 1.6915 / 1.1474）。
    以下为 149 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格与区间中点中位数偏离 十分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    149 在 148 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_mid : 区间中点中位数偏离 ln(最近 8 个窗口 (最高 + 最低) / 2 的中位数 / 当前价)。
    r_med 的锚点只用每个窗口末的一个报价，r_mid 用的是每个窗口内全部成交覆盖的价格区间的中点，
    反映窗口内的成交中枢，而不只是窗口收盘时刻的价格。窗口长度沿用已有的 8，无新增参数。10 分量 x 四网络 = 40 列做 IQR Winsor 均值。
    这一分量来自第十一批“稳健锚点”信号的分量级筛选（逐个作为新分量放进 148 的 IQR），
    4 个中只有它验证段与保留段同时为正；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4303 / 0.4236（148 0.4365 / 0.4285）；
      IC 0.07995 / 0.06575（148 0.07984 / 0.06590）；
      ICIR 1.6975 / 1.1403（148 1.6944 / 1.1434）；
      Sharpe 47.31 / 34.16（148 47.28 / 33.82）；
      压力 1.6915 / 1.1474（148 1.6929 / 1.1516）。
    以下为 148 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 + 价格中位数偏离 九分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    148 在 147 号基础上只在 IQR 内部加一个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_med : 价格中位数偏离 ln(最近 8 个窗口末价的中位数 / 当前价)。
    阶梯中的 r_rv8 以同样 8 个窗口的成交量加权均价为锚，会被单个放量窗口拉动；中位数锚点不受成交量和单个极端价格影响，
    衡量的是当前价相对近期“典型价格”的偏离。窗口长度沿用阶梯中已有的 8，无新增参数。9 分量 x 四网络 = 36 列做 IQR Winsor 均值。
    这一分量来自第十批 16 个“换锚点”反转信号的分量级筛选（逐个作为新分量放进 147 的 IQR），
    16 个中只有它验证段与保留段同时为正；所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4365 / 0.4285（147 0.4389 / 0.4314）；
      IC 0.07984 / 0.06590（147 0.07933 / 0.06569）；
      ICIR 1.6944 / 1.1434（147 1.6821 / 1.1417）；
      Sharpe 47.28 / 33.82（147 46.51 / 33.45）；
      压力 1.6929 / 1.1516（147 1.6729 / 1.1436）。
    以下为 147 的说明。
    更长倍增阶梯 + 日内低点距离 + VWAP 斜率 八分量四网络同伴排名 IQR + 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    147 在 146 号基础上只在 IQR 内部加两个反转分量（同伴网络、中性化、聚合、慢特征融合全部不变）：
      r_lo    : 当日低点距离 ln(min(当日累计窗口最低价, 昨收) / 当前价)，衡量从当日低点（或昨收）以来的反弹幅度；
      r_slope : 窗口 VWAP 斜率 ln(滚动 8 窗口 VWAP / 当前窗口 VWAP)，衡量当前窗口成交均价相对近期均价的抬升。
    两者与阶梯中的“价格相对 VWAP 偏离”口径不同：前者以当日极值为锚，后者只看成交均价、不含当前报价，
    与已有分量相关较低。8 分量 x 四网络 = 32 列做 IQR Winsor 均值。
    这两个分量来自对 215 个结构不同信号的批量分量级筛选（逐个作为新分量放进 146 的 IQR），验证段与保留段同时为正；
    所有候选统一用同一套网络与聚合，未做参数调整。
      验证 / 保留两段换手 0.4389 / 0.4314（146 0.4798 / 0.4698）；
      IC 0.07933 / 0.06569（146 0.08049 / 0.06665）；
      ICIR 1.6821 / 1.1417（146 1.7022 / 1.1562）；
      Sharpe 46.51 / 33.45（146 47.27 / 34.23）；
      压力 1.6729 / 1.1436（146 1.6992 / 1.1468）。
    以下为 146 的说明。
    更长倍增阶梯四网络同伴排名 IQR（144）+ 慢特征组合（低波动 / 小盘 / 低价），截面排名 1 : 0.5 融合。
    146 在 144 号基础上只加一个与反转正交的慢特征组合：
      c_park  : 低波动，最近 40 个窗口 Parkinson 区间方差 ln(窗口最高 / 最低)^2 的均值取负；
      c_size  : 小盘，-ln(1 + 最近 40 个窗口清洗后成交额之和)（144 中已有，用于中性化）；
      c_lowpx : 低价，-ln(当前价)（144 中已有，用于中性化）。
    三者截面排名等权相加，再与 144 的 IQR 输出按固定的 1 : 0.5 做截面排名融合。三者都是 A 股文献中的经典异象
    （低波动、小市值、低价），与反转主信号的风格剔除后相关约为 0；它们几乎不随时间变化，融合后换手下降。
    这一组合来自对 130 个结构不同信号的批量筛选，验证段与保留段同时为正；融合权重对全部信号统一取 0.5，未逐个调参。
      验证 / 保留两段换手 0.4798 / 0.4698（144 0.4969 / 0.4883）；
      IC 0.08049 / 0.06665（144 0.07809 / 0.06452）；
      ICIR 1.7022 / 1.1562（144 1.6088 / 1.1070）；
      Sharpe 47.27 / 34.23（144 49.77 / 35.09）；
      压力 1.6992 / 1.1468（144 1.5654 / 1.0782）。
    以下为 144 的说明。
    平台原始价格口径下的更长倍增阶梯超额四网络同伴排名，IQR Winsor 均值聚合。

    数据读取、窗口末价格和底层信号完全沿用平台高分
    jyc_peer_component_iqr_winsor_v1，不纳入 09:25 竞价快照，也不做窗口末价格兜底。
    144 在 143 号基础上只加一个分量：倍增阶梯沿同一倍增序列向慢端再延一档，加入滚动 32 窗口（约 4 个交易日，
    接近一周）清洗后成交 VWAP 偏离 r_rv32。六个分量为 2 / 4 / 8 / 16 / 32 窗口 VWAP 偏离 + 日内反转，
    x 四张同伴网络，24 列做 IQR Winsor 均值。周尺度锚点更新更慢，合成信号相邻截面更连续，换手继续下降；
    窗口长度仍是同一倍增序列的下一项，无新增参数。
      验证 / 保留两段换手 0.4969 / 0.4883（143 0.5223 / 0.5155）；
      IC 0.07809 / 0.06452（143 0.07838 / 0.06514）；
      ICIR 1.6088 / 1.1070（143 1.6135 / 1.1239）；
      Sharpe 49.77 / 35.09（143 50.83 / 35.84）；
      压力 1.5654 / 1.0782（143 1.5519 / 1.0812）。
    以下为 143 的说明。
    143 在 142 号基础上只加一个分量：倍增阶梯沿同一倍增序列向慢端再延一档，加入滚动 16 窗口（约 2 个交易日）
    清洗后成交 VWAP 偏离 r_rv16。五个分量为 2 / 4 / 8 / 16 窗口 VWAP 偏离 + 日内反转，x 四张同伴网络，
    20 列做 IQR Winsor 均值。慢端锚点跨日更新缓慢，加入后合成信号的截面变化更平稳，同时多日尺度的
    VWAP 偏离本身也带有反转信息；窗口长度仍是同一倍增序列的下一项，无新增参数。
      验证 / 保留两段换手 0.5223 / 0.5155（142 0.5688 / 0.5623）；
      IC 0.07838 / 0.06514（142 0.07808 / 0.06547）；
      ICIR 1.6135 / 1.1239（142 1.6142 / 1.1341）；
      Sharpe 50.83 / 35.84（142 51.17 / 36.54）；
      压力 1.5519 / 1.0812（142 1.5298 / 1.0755）。
    以下为 142 的说明。
    142 在 140 号基础上只换一个分量：30 分钟反转 r_30（只看上一窗口末的单个价格点）换成滚动 2 窗口（1 小时）
    清洗后成交 VWAP 偏离 r_rv2。四个分量成为 2 / 4 / 8 窗口 VWAP 偏离的倍增阶梯 + 日内反转，尺度均匀覆盖，
    且短端由成交量加权的均价代替单点价格，相邻截面之间更连续；首窗不再与 r_day 重复，无需再做首窗去重。
    各分量 x 四张同伴网络，16 列做 IQR Winsor 均值。窗口长度沿用已有的倍增序列，无新增参数。
      验证 / 保留两段换手 0.5688 / 0.5623（140 0.6005 / 0.5949）；
      IC 0.07808 / 0.06547（140 0.07906 / 0.06626）；
      ICIR 1.6142 / 1.1341（140 1.6303 / 1.1475）；
      Sharpe 51.17 / 36.54（140 51.55 / 36.53）；
      压力 1.5298 / 1.0755（140 1.5472 / 1.0809）。
    以下为 141 的说明（141 去掉 r_30 只留三个慢分量，换手降得更多但 IC 也掉得更多，142 是两者之间的折中）。
    141 在 140 号基础上只做一个无参结构改动：去掉 30 分钟反转分量 r_30，只保留滚动 VWAP 4/8 窗口偏离与
    日内反转三个慢分量（各 x 四张同伴网络，12 列做 IQR Winsor 均值）。r_30 只看最近一个窗口，相邻截面之间
    几乎完全更新，是多空两腿成分替换的主要来源；平台五项得分中换手的权重最大，r_30 的边际 IC 不足以抵消它带来的换手。
    r_30 仍用于构建 30 分钟同伴网络（窗口收益相关），只是不再作为合成分量。无新增参数。
      验证 / 保留两段换手 0.4775 / 0.4696（140 0.6005 / 0.5949，降约 0.12）；
      IC 0.07646 / 0.06347（140 0.07906 / 0.06626）；
      ICIR 1.5766 / 1.1041（140 1.6303 / 1.1475）；
      Sharpe 49.52 / 34.85（140 51.55 / 36.53）；
      压力 1.5145 / 1.0732（140 1.5472 / 1.0809）。
    以下为 140 的说明。
    140 在 137 号基础上只改最后一步聚合：16 列中性化排名由 Midhinge
    （25%/75% 分位点均值）改为 IQR Winsor 均值（各列截到本行四分位区间后取均值）。Midhinge 只由两个分位点决定，
    任一列的小幅变动都可能让分位点在相邻列之间跳动，截面排名随之抖动；Winsor 均值让中间一半的列都连续地参与，
    相邻截面更平滑。无新增参数。平台五项得分对换手最敏感之一，此改动的主要收益是降换手：
      开发 / 验证 / 保留三段换手 0.6009 / 0.6005 / 0.5949（137 0.6211 / 0.6194 / 0.6132，三段均降约 0.02）；
      IC 0.08895 / 0.07906 / 0.06626（137 0.08917 / 0.07937 / 0.06643）；
      ICIR 1.4783 / 1.6303 / 1.1475（137 1.4816 / 1.6362 / 1.1496）；
      Sharpe 49.77 / 51.55 / 36.53（137 50.05 / 52.27 / 36.73）；
      压力 1.4021 / 1.5472 / 1.0809（137 1.4049 / 1.5532 / 1.0807）。
    以下为 137 的说明。
    在 136 号（首窗去重的 4 个反转分量 x 日频/30 分钟两张同伴网络，各做“中位数残差 + 同伴内排名”融合与
    四特征中性化，8 列取 Midhinge）基础上只加一个无参结构改动：
      另建两张“超额收益”同伴网络：日收益 / 30 分钟窗口收益先减去同一时点的截面均值，再按相关性取前 30 名同伴。
      原始收益相关里很大一部分是共同的市场涨跌，高贝塔股票会因此互为同伴；超额网络只保留真实的同类联动。
      原始 / 超额 x 日频 / 30 分钟四张网络并列，每个分量在每张网络上照旧做融合与中性化，16 列取 Midhinge
      （09:30 截面去掉 r_30 的 4 列，用 12 列）。不新增任何参数，同伴数、回看期、最少观测都与 136 相同。
    相对 136，本地开发 / 验证 / 保留三段：
      IC 0.08917 / 0.07937 / 0.06643（136 0.08910 / 0.07944 / 0.06629）；
      ICIR 1.4816 / 1.6362 / 1.1496（136 1.4692 / 1.6274 / 1.1277）；
      Sharpe 50.05 / 52.27 / 36.73（136 49.94 / 51.94 / 36.93）；
      压力 1.4049 / 1.5532 / 1.0807（136 1.3931 / 1.5425 / 1.0629）；
      换手 0.6211 / 0.6194 / 0.6132（136 0.6188 / 0.6175 / 0.6102）。
    136 的改动：当日首个窗口的 30 分钟反转以昨收为基准，与 r_day 完全相同，首窗把 r_30 各列置为缺失，避免重复计票；
    r_30 仍保留首窗定义用于构建 30 分钟同伴网络。
    公共构造（103–107 共用同一段代码，只有最后的合成方式不同）：
      反转分量（与 90 号相同，均只用过去数据）：
        r_30  : 30 分钟反转 ln(上一窗口末价 / 当前价)，当日首个窗口以昨收为基准（57 号）
        r_rv4 / r_rv8 : 滚动 4/8 个窗口（2/4 小时，跨日）清洗后成交 VWAP 偏离 ln(滚动 VWAP / 当前价)
        r_day : 日内反转 ln(昨收 / 当前价)（56 号）
      同伴（伪行业）残差：每个自然月月初，用此前 60 个交易日的日收益 ln(收盘段价格 / 昨收) 计算股票池内两两相关系数，
        每只股票取相关性最高的 30 只为同伴（至少 30 个共同观测；只用过去数据）。分量残差 = 分量 - 同伴中位数
        （同伴有效值不足 3 个时减截面均值）。思路：行业内反转强于全市场反转（Da, Liu & Schaumburg 2014），
        去掉“整组一起涨跌”的部分后，只保留个股相对同类股票的过度反应。
      四特征中性化：因子截面排名对 规模（-ln 最近 40 窗口成交额）、特质波动、非流动性（Amihud）、低价 四个特征的截面排名
        做逐截面回归，取残差。反转信号天然偏向小盘、高波动股票，而这些暴露本身会拖累排序；剔除后 IC、IC_IR、多空夏普同时提升。
      数据清洗：成交均价落在该分钟 [最低价 x 0.98, 最高价 x 1.02] 之外的分钟不计入任何 VWAP / 成交额累加。

    稳健性：最终缺失补 0（中性），每个截面缺失率为 0；向前取不到足够历史时同伴退化为截面均值、特征排名记为中性。
    性能：纯 SQL 向量化（单次 GROUP BY，无窗口函数、无 Python UDF）；相关矩阵每月计算一次（约 1200 x 1200），
      同伴均值按月份用数组索引批量计算。

    评测时平台会注入 datasources、start_date 和 end_date。
    返回且仅返回 ['date', 'instrument', 'factor'] 三列。
    """
    import numpy as np
    import pandas as pd
    import dai

    bar1m = datasources["bar1m"]

    # 向前多取数据：同伴相关系数需要月初之前 60 个交易日的日收益（约 90 个自然日，加上月内偏移与长假留足余量）。
    LOOKBACK_DAYS = 150
    PEER_K, PEER_LOOKBACK, PEER_MIN_OBS = 30, 60, 30
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=LOOKBACK_DAYS)).strftime("%Y-%m-%d %H:%M:%S")

    sql = f"""
    WITH cte_raw AS (
        SELECT
            instrument_id,
            strftime(date, '%Y-%m-%d') AS trading_day,
            strftime(date, '%H%M%S') AS hms,
            CAST(COALESCE(bid_price1, 0) AS DOUBLE) AS bp1,
            CAST(COALESCE(ask_price1, 0) AS DOUBLE) AS ap1,
            CAST(COALESCE(ask_volume1, 0) AS DOUBLE) AS av1,
            CAST(COALESCE(ask_volume2, 0) AS DOUBLE) AS av2,
            CAST(COALESCE(ask_volume3, 0) AS DOUBLE) AS av3,
            CAST(COALESCE(bid_volume1, 0) AS DOUBLE) AS bv1,
            CAST(COALESCE(bid_volume2, 0) AS DOUBLE) AS bv2,
            CAST(COALESCE(bid_volume3, 0) AS DOUBLE) AS bv3,
            CAST(COALESCE(deal_number, 0) AS DOUBLE) AS deal,
            CAST(COALESCE(high, 0) AS DOUBLE) AS px_high,
            CAST(COALESCE(low, 0) AS DOUBLE) AS px_low,
            CAST(COALESCE(close, 0) AS DOUBLE) AS px_close,
            CAST(COALESCE(pre_close, 0) AS DOUBLE) AS pre_close,
            CAST(COALESCE(volume, 0) AS DOUBLE) AS vol,
            CAST(COALESCE(amount, 0) AS DOUBLE) AS amt
        FROM {bar1m}
    ),
    cte_snapshot AS (
        SELECT
            instrument_id,
            trading_day,
            hms,
            pre_close,
            px_high,
            px_low,
            ap1,
            av1,
            av2,
            av3,
            bp1,
            bv1,
            bv2,
            bv3,
            px_close,
            deal,
            CASE
                WHEN vol > 0 AND amt > 0 AND px_low > 0
                     AND amt / vol >= px_low * 0.98 AND amt / vol <= px_high * 1.02
                THEN amt ELSE 0
            END AS amt_c,
            CASE
                WHEN vol > 0 AND amt > 0 AND px_low > 0
                     AND amt / vol >= px_low * 0.98 AND amt / vol <= px_high * 1.02
                THEN vol ELSE 0
            END AS vol_c,
            CASE
                WHEN bp1 > 0 AND ap1 > bp1 THEN (bp1 + ap1) / 2.0
                WHEN px_close > 0 THEN px_close
                ELSE 0
            END AS px,
            CASE
                WHEN hms > '090000' AND hms <= '093000' THEN 93000
                WHEN hms > '093000' AND hms <= '100000' THEN 100000
                WHEN hms > '100000' AND hms <= '103000' THEN 103000
                WHEN hms > '103000' AND hms <= '110000' THEN 110000
                WHEN hms > '110000' AND hms <= '113000' THEN 113000
                WHEN hms > '130000' AND hms <= '133000' THEN 133000
                WHEN hms > '133000' AND hms <= '140000' THEN 140000
                WHEN hms > '140000' AND hms <= '143000' THEN 143000
                WHEN hms > '143000' AND hms <= '150000' THEN 150000
                ELSE -1
            END AS time_segment
        FROM cte_raw
    ),
    cte_window AS (
        SELECT
            trading_day, time_segment, instrument_id,
            SUM(amt_c) AS sum_amt,
            SUM(vol_c) AS sum_vol,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN px ELSE 0 END) AS last_px,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN ap1 ELSE 0 END) AS last_ap1,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN av1 ELSE 0 END) AS last_av1,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN av2 ELSE 0 END) AS last_av2,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN av3 ELSE 0 END) AS last_av3,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN bp1 ELSE 0 END) AS last_bp1,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN bv1 ELSE 0 END) AS last_bv1,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN bv2 ELSE 0 END) AS last_bv2,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN bv3 ELSE 0 END) AS last_bv3,
            MAX(CASE WHEN CAST(hms AS INTEGER) = time_segment THEN px_close ELSE 0 END) AS last_close,
            SUM(deal) AS sum_deal,
            MAX(pre_close) AS pre_close,
            MAX(px_high) AS bar_high,
            MIN(CASE WHEN px_low > 0 THEN px_low END) AS bar_low
        FROM cte_snapshot
        WHERE time_segment != -1
        GROUP BY instrument_id, trading_day, time_segment
    )
    SELECT
        CAST(CONCAT(
            f.trading_day,
            ' ',
            strftime(strptime(LPAD(f.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        f.sum_amt, f.sum_vol, f.last_px, f.pre_close, f.bar_high, f.bar_low,
        f.last_ap1, f.last_av1, f.last_av2, f.last_av3, f.last_bp1, f.last_bv1, f.last_bv2, f.last_bv3, f.last_close, f.sum_deal
    FROM cte_window f
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={"date": [query_start, end_date]}, compression=True).df()
    df = df.dropna(subset=["instrument"]).drop_duplicates(["date", "instrument"])
    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    for c in ["sum_amt", "sum_vol", "last_px", "pre_close", "bar_high", "bar_low",
              "last_ap1", "last_av1", "last_av2", "last_av3", "last_bp1", "last_bv1", "last_bv2", "last_bv3", "last_close", "sum_deal"]:
        df[c] = df[c].astype("float64")
    df["last_px"] = df["last_px"].where(df["last_px"] > 0)
    # 161：窗口末盘口 / 成交价在没有当前价（09:30 截面）时一律缺失（盘口为空记 0，由各分量自行判断有效性）。
    for c in ["last_ap1", "last_av1", "last_av2", "last_av3", "last_bp1", "last_bv1", "last_bv2", "last_bv3", "last_close"]:
        df[c] = df[c].where(df["last_px"].notna())
    df["pre_close"] = df["pre_close"].where(df["pre_close"] > 0)

    # 股票池：同伴只在股票池内挑选（向前取历史区间的成分，取不到时用评估区间的成分）。
    instruments_table = datasources.get("instruments", "cpt_jyc_2026_instruments")
    pool_hist = dai.query(
        f"SELECT date, instrument FROM {instruments_table}",
        filters={"date": [query_start, end_date]},
        compression=True,
    ).df()
    pool_hist["date"] = pd.to_datetime(pool_hist["date"])
    pool_hist["instrument"] = pool_hist["instrument"].astype(str)
    stk_pool = pool_hist[(pool_hist["date"] >= pd.to_datetime(start_date)) & (pool_hist["date"] <= pd.to_datetime(end_date))]
    universe = set(pool_hist["instrument"].unique())
    df = df[df["instrument"].isin(universe)]
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
    df["day"] = df["date"].dt.normalize()

    def safe_log_ratio(num, den):
        ok = (num > 0) & (den > 0)
        return pd.Series(np.where(ok, np.log(num.where(ok, 1.0) / den.where(ok, 1.0)), np.nan), index=num.index)

    def vwap_of(amt, vol):
        return (amt / vol.where(vol > 0)).where(amt > 0)

    def roll(s, k, fn, min_periods):
        return s.groupby(df["instrument"], sort=False).transform(lambda x: getattr(x.rolling(k, min_periods=min_periods), fn)())

    grp = df.groupby("instrument", sort=False)
    grp_day = df.groupby(["instrument", "day"], sort=False)
    same_day = grp["day"].shift(1) == df["day"]
    px = df["last_px"]

    day_vwap = vwap_of(grp_day["sum_amt"].cumsum(), grp_day["sum_vol"].cumsum())
    r_dvwap = safe_log_ratio(day_vwap, px)

    def rolling_vwap_rev(k):
        return safe_log_ratio(vwap_of(roll(df["sum_amt"], k, "sum", 1), roll(df["sum_vol"], k, "sum", 1)), px).fillna(r_dvwap)

    df["r_30"] = safe_log_ratio(grp["last_px"].shift(1).where(same_day, df["pre_close"]), px)
    df["r_rv2"] = rolling_vwap_rev(2)
    df["r_rv4"] = rolling_vwap_rev(4)
    df["r_rv8"] = rolling_vwap_rev(8)
    df["r_rv16"] = rolling_vwap_rev(16)
    df["r_rv32"] = rolling_vwap_rev(32)
    df["r_day"] = safe_log_ratio(df["pre_close"], px)
    # 148：价格中位数偏离 ln(最近 8 个窗口末价的中位数 / 当前价)，以稳健的近期价格中枢为锚。
    df["r_med"] = safe_log_ratio(roll(px, 8, "median", 2), px)
    # 149：区间中点中位数偏离 ln(最近 8 个窗口 (最高 + 最低) / 2 的中位数 / 当前价)，以窗口内全部成交区间的中枢为锚。
    df["r_mid"] = safe_log_ratio(roll(((df["bar_high"] + df["bar_low"]) / 2).where(df["bar_low"] > 0), 8, "median", 2), px)
    # 150：最低价中位数偏离 ln(最近 8 个窗口最低价的中位数 / 当前价)，以近期各窗口低点的稳健中枢为支撑锚。
    df["r_lomed"] = safe_log_ratio(roll(df["bar_low"].where(df["bar_low"] > 0), 8, "median", 2), px)
    # 151：低振幅加权价格锚点偏离，最近 8 个窗口末价按 1 / (ln(最高 / 最低) + 0.001) 加权平均，振幅小的窗口权重大。
    calm_w = 1.0 / (safe_log_ratio(df["bar_high"], df["bar_low"]).where(df["bar_high"] > df["bar_low"]) + 1e-3)
    calm_den = roll(calm_w.where(px > 0), 8, "sum", 2)
    df["r_calm"] = safe_log_ratio(roll(px.where(px > 0) * calm_w, 8, "sum", 2) / calm_den.where(calm_den > 0), px)
    # 161：新增分量的公共工具。
    def lag(s, k):
        return s.groupby(df["instrument"], sort=False).shift(k)

    def B(c):
        return df["last_" + c]

    def dvwap():
        g = df.groupby([df["instrument"], df["day"]], sort=False)
        return vwap_of(g["sum_amt"].cumsum(), g["sum_vol"].cumsum())

    def side_px3(x):
        """单侧三档挂单量加权报价（x 为 "b" 或 "a"）。"""
        v = sum(B(f"{x}v{i}") for i in (1, 2, 3))
        return (sum(B(f"{x}p{i}") * B(f"{x}v{i}") for i in (1, 2, 3)) / v.where(v > 0)).where(book_ok())

    def book_ok():
        return (B("bp1") > 0) & (B("ap1") >= B("bp1"))

    def qmid():
        return ((B("bp1") + B("ap1")) / 2).where(book_ok())

    def micro():
        bv, av = B("bv1"), B("av1")
        return ((B("bp1") * av + B("ap1") * bv) / (bv + av).where(bv + av > 0)).where(book_ok())

    def depth3(sides=("bv", "av")):
        s = sum(B(f"{x}{i}") for x in sides for i in (1, 2, 3))
        return s.where(s > 0)

    def book3():
        num = sum(B(f"bp{i}") * B(f"bv{i}") + B(f"ap{i}") * B(f"av{i}") for i in (1, 2, 3))
        return num / depth3()

    def rvwap(k):
        return vwap_of(roll(df["sum_amt"], k, "sum", 1), roll(df["sum_vol"], k, "sum", 1))

    def calm_w():
        return 1.0 / (safe_log_ratio(df["bar_high"], df["bar_low"]).where(df["bar_high"] > df["bar_low"]) + 1e-3)

    def wanchor_of(q, w, n=8):
        q = q.where(q > 0)
        ok = q.notna() & w.notna() & (w > 0)
        den = roll(w.where(ok), n, "sum", 2)
        return safe_log_ratio(roll((q * w).where(ok), n, "sum", 2) / den.where(den > 0), q)

    def self_anchor(q, fn="mean", n=8):
        q = q.where(q > 0)
        return safe_log_ratio(roll(q, n, fn, 2), q)

    def px_at_max(key, n=8):
        """最近 n 个窗口中 key 最大那个窗口的成交 VWAP（至少 2 个有效窗口）。"""
        vw = vwap_of(df["sum_amt"], df["sum_vol"])
        ks = np.column_stack([lag(key, k).to_numpy(np.float64) for k in range(n)])
        vs = np.column_stack([lag(vw, k).to_numpy(np.float64) for k in range(n)])
        ks = np.where(np.isfinite(ks) & np.isfinite(vs) & (vs > 0), ks, -np.inf)
        j = ks.argmax(1)
        out = vs[np.arange(len(j)), j]
        return pd.Series(np.where(np.isfinite(ks).sum(1) >= 2, out, np.nan), index=df.index)

    def day_mean(x):
        """当日截至当前窗口 x 的累计均值（跳过缺失）。"""
        k = [df["instrument"], df["day"]]
        n = x.notna().astype(np.float64).groupby(k, sort=False).cumsum()
        return x.fillna(0).groupby(k, sort=False).cumsum() / n.where(n > 0)

    def day_first(x):
        """当日 x 的第一个有效值。"""
        return x.groupby([df["instrument"], df["day"]], sort=False).transform("first")
    # 161：ln(最近 8 个窗口末一档中间价均值 / 当前一档中间价)，涨跌停单边盘口的窗口不参与锚点、当期缺失，没有当前价时缺失。
    df["r_qmid"] = ((self_anchor(qmid()))).where(px.notna())
    # 161：最近 40 个窗口 ln(窗口末当前价 / 窗口末成交价) 的均值（成交持续落在买一侧即卖压时为负）；方向取负，没有当前价时缺失。
    df["r_cmid"] = (-(roll(safe_log_ratio(px, B("close")), 40, "mean", 10))).where(px.notna())
    # 161：ln(最近 8 个窗口末价按三档挂单总量加权的均价 / 当前价)，挂单厚的窗口价格更可信，没有当前价时缺失。
    df["r_deep"] = ((wanchor_of(px, depth3()))).where(px.notna())
    # 161：ln(最近 8 个窗口成交 VWAP / 当前一档微价格)，微价格按对侧挂单量加权买一 / 卖一价，没有当前价时缺失。
    df["r_micro"] = ((safe_log_ratio(rvwap(8), micro()))).where(px.notna())
    # 161：ln(最近 8 个窗口末价按窗口成交笔数加权的均价 / 当前价)，没有当前价时缺失。
    df["r_dealpx"] = ((wanchor_of(px, df["sum_deal"].where(df["sum_deal"] > 0)))).where(px.notna())

    # 个股 beta（40 窗口滚动回归，截断 [-1, 4]，不足取 1），用于特质波动。
    win_ret = safe_log_ratio(px, grp["last_px"].shift(1))
    ret30 = safe_log_ratio(px, grp["last_px"].shift(1).where(same_day)).fillna(safe_log_ratio(px, df["pre_close"]))
    mkt30 = ret30.groupby(df["date"]).transform("mean")
    ex, ey = roll(ret30, 40, "mean", 10), roll(mkt30, 40, "mean", 10)
    exy, ey2 = roll(ret30 * mkt30, 40, "mean", 10), roll(mkt30 ** 2, 40, "mean", 10)
    var_m = (ey2 - ey ** 2).where((ey2 - ey ** 2) > 1e-12)
    beta = ((exy - ex * ey) / var_m).clip(-1, 4).fillna(1.0)

    # 股票特征：规模（-ln 最近 40 窗口成交额）、特质波动（取负）、非流动性（Amihud）、低价、MAX（取负）。
    df["c_size"] = -np.log1p(roll(df["sum_amt"], 40, "sum", 10))
    idio = win_ret - beta * win_ret.groupby(df["date"]).transform("mean")
    df["c_ivol"] = -roll(idio, 40, "std", 10)
    df["c_illiq"] = roll(win_ret.abs() / (df["sum_amt"] / 1e8).where(df["sum_amt"] > 0), 40, "mean", 10)
    df["c_lowpx"] = -np.log(px.where(px > 0))
    df["c_max"] = -roll(win_ret, 40, "max", 10)
    # 低波动（取负）：最近 40 个窗口 Parkinson 区间方差 ln(窗口最高 / 最低)^2 的均值。
    df["c_park"] = -roll(safe_log_ratio(df["bar_high"], df["bar_low"]) ** 2, 40, "mean", 10)

    # 同伴：每个自然月月初，用此前 PEER_LOOKBACK 个交易日的收益计算相关系数，两种频率各建一张网络：
    #   日频：日收益（收盘段价格 / 昨收），至少 PEER_MIN_OBS 个共同观测；
    #   30 分钟：窗口收益（上一窗口末价，当日首个窗口用昨收），取回看期内全部窗口，至少 4 x PEER_MIN_OBS 个共同观测。
    daily = df.groupby(["day", "instrument"], sort=False).agg(px_last=("last_px", "last"), pc=("pre_close", "first"))
    daily_ret = safe_log_ratio(daily["px_last"], daily["pc"]).unstack("instrument").sort_index()
    peer_insts = daily_ret.columns.values
    all_days = daily_ret.index
    day_month = all_days.to_period("M")
    out_months = pd.to_datetime(stk_pool["date"]).dt.normalize().dt.to_period("M").unique()
    win_ret30 = pd.Series(
        (-df["r_30"]).values, index=pd.MultiIndex.from_arrays([df["date"].values, df["instrument"].values])
    ).unstack().sort_index().reindex(columns=peer_insts)

    def build_peers(ret_wide, row_day, need):
        """返回 {月份: 同伴下标矩阵}；ret_wide 每行是一个收益观测，row_day 为该行所属交易日。"""
        peers = {}
        for m in out_months:
            in_month = all_days[day_month == m]
            if len(in_month) == 0:
                continue
            prior = all_days[all_days < in_month[0]][-PEER_LOOKBACK:]
            if len(prior) < PEER_MIN_OBS:
                continue
            hist = ret_wide[(row_day < in_month[0]) & (row_day >= prior[0])]
            ok = (hist.notna().sum() >= need).values
            corr = hist.loc[:, ok].corr(min_periods=need).values
            np.fill_diagonal(corr, -np.inf)
            corr = np.nan_to_num(corr, nan=-np.inf)
            sub = np.where(ok)[0]
            idx = np.full((len(peer_insts), PEER_K), -1, dtype=np.int64)
            idx[sub] = sub[np.argsort(-corr, axis=1, kind="stable")[:, :PEER_K]]
            peers[m] = idx
        return peers

    # 超额网络：每个收益观测先减去同一时点的截面均值，再按超额收益的相关性挑同伴（去掉市场共振带来的伪同伴）。
    peers_by_freq = {
        "day": build_peers(daily_ret, all_days, PEER_MIN_OBS),
        "30": build_peers(win_ret30, win_ret30.index.normalize(), PEER_MIN_OBS * 4),
        "day_excess": build_peers(daily_ret.sub(daily_ret.mean(axis=1), axis=0), all_days, PEER_MIN_OBS),
        "30_excess": build_peers(win_ret30.sub(win_ret30.mean(axis=1), axis=0), win_ret30.index.normalize(), PEER_MIN_OBS * 4),
    }

    df = df.replace([np.inf, -np.inf], np.nan)
    keep = ["r_rv2", "r_rv4", "r_rv8", "r_rv16", "r_rv32", "r_day", "r_med", "r_mid", "r_lomed", "r_calm", "r_qmid", "r_cmid", "r_deep", "r_micro", "r_dealpx", "c_size", "c_ivol", "c_illiq", "c_lowpx", "c_max", "c_park"]
    result = pd.merge(df[["date", "instrument"] + keep], stk_pool[["date", "instrument"]], how="inner", on=["date", "instrument"])
    result = result.drop_duplicates(["date", "instrument"]).sort_values(["date", "instrument"]).reset_index(drop=True)
    result_month = result["date"].dt.to_period("M")

    def peer_mean(col, peers_by_month):
        """每个截面上同伴的 col 中位数（至少 3 个有效值），没有同伴信息时为缺失。"""
        out = np.full(len(result), np.nan)
        for m, rows in result.groupby(result_month, sort=False).indices.items():
            idx = peers_by_month.get(m)
            if idx is None:
                continue
            sub = result.iloc[rows]
            wide = sub.pivot(index="date", columns="instrument", values=col).reindex(columns=peer_insts)
            valid = idx >= 0
            g = wide.values[:, np.where(valid, idx, 0)]
            g = np.where(valid[None, :, :], g, np.nan)
            cnt = (~np.isnan(g)).sum(axis=2)
            pm = np.where(cnt >= 3, np.nanmedian(g, axis=2), np.nan)
            r_pos = wide.index.get_indexer(sub["date"])
            c_pos = pd.Index(peer_insts).get_indexer(sub["instrument"])
            out[rows] = np.where(c_pos >= 0, pm[r_pos, np.maximum(c_pos, 0)], np.nan)
        return pd.Series(out, index=result.index)

    def peer_rank(col, peers_by_month):
        """个股 col 在其同伴中的相对排名（低于自身的同伴占比，平局记半），居中到 [-0.5, 0.5]；
        同伴有效值不足 3 个或没有同伴信息时回退为截面居中排名。"""
        out = np.full(len(result), np.nan)
        for m, rows in result.groupby(result_month, sort=False).indices.items():
            idx = peers_by_month.get(m)
            if idx is None:
                continue
            sub = result.iloc[rows]
            wide = sub.pivot(index="date", columns="instrument", values=col).reindex(columns=peer_insts)
            x = wide.values
            valid = idx >= 0
            g = x[:, np.where(valid, idx, 0)]
            g = np.where(valid[None, :, :], g, np.nan)
            own = x[:, :, None]
            ok = ~np.isnan(g) & ~np.isnan(own)
            cnt = ok.sum(axis=2)
            score = ((g < own) & ok).sum(axis=2) + 0.5 * ((g == own) & ok).sum(axis=2)
            rel = np.where(cnt >= 3, score / np.maximum(cnt, 1) - 0.5, np.nan)
            r_pos = wide.index.get_indexer(sub["date"])
            c_pos = pd.Index(peer_insts).get_indexer(sub["instrument"])
            out[rows] = np.where(c_pos >= 0, rel[r_pos, np.maximum(c_pos, 0)], np.nan)
        fallback = result.groupby("date")[col].rank(pct=True) - 0.5
        return pd.Series(out, index=result.index).fillna(fallback)

    def rank_blend(cols, weights=None):
        """各分量在股票池内逐截面居中排名后加权平均（缺失分量不参与，权重按可用分量重新归一）。"""
        weights = weights or [1.0] * len(cols)
        ranks = (result.groupby("date")[cols].rank(pct=True) - 0.5).to_numpy()
        w = np.array(weights, dtype=float)
        has = ~np.isnan(ranks)
        den = (has * w).sum(axis=1)
        num = np.nansum(np.nan_to_num(ranks) * w, axis=1)
        return pd.Series(np.where(den > 0, num / np.where(den > 0, den, 1.0), np.nan), index=result.index)

    def neutralize(col, chars, frac=1.0):
        """截面中性化：col 的居中排名对各特征居中排名（缺失记 0）逐截面 OLS（含截距），返回 排名 - frac x 拟合值。"""
        y = result.groupby("date")[col].rank(pct=True) - 0.5
        X = np.column_stack([(result.groupby("date")[c].rank(pct=True) - 0.5).fillna(0.0).values for c in chars])
        ok = y.notna().values
        codes, uniq = pd.factorize(result["date"].values)
        k, n_grp = X.shape[1], len(uniq)
        Xc = np.column_stack([np.ones(len(result)), X]) * ok[:, None]
        yv = np.nan_to_num(y.values) * ok
        xtx = np.zeros((n_grp, k + 1, k + 1))
        xty = np.zeros((n_grp, k + 1))
        for i in range(k + 1):
            xty[:, i] = np.bincount(codes, Xc[:, i] * yv, minlength=n_grp)
            for j in range(i, k + 1):
                v = np.bincount(codes, Xc[:, i] * Xc[:, j], minlength=n_grp)
                xtx[:, i, j] = v
                xtx[:, j, i] = v
        b = np.linalg.solve(xtx + 1e-9 * np.eye(k + 1), xty[..., None])[..., 0]
        fit = (X * b[codes, 1:]).sum(axis=1)
        return pd.Series(np.where(ok, y.values - frac * fit, np.nan), index=result.index)

    CHARS4 = ["c_size", "c_ivol", "c_illiq", "c_lowpx"]
    neutral_cols = []
    for c in ["r_rv2", "r_rv4", "r_rv8", "r_rv16", "r_rv32", "r_day", "r_med", "r_mid", "r_lomed", "r_calm", "r_qmid", "r_cmid", "r_deep", "r_micro", "r_dealpx"]:
        cross_mean = result.groupby("date")[c].transform("mean")
        for f, peers in peers_by_freq.items():
            # 每个分量 x 每张同伴网络：同伴中位数残差与同伴内相对排名各自截面排名后等权融合，再做四特征中性化。
            result[f"{c}_p{f}"] = result[c] - peer_mean(c, peers).fillna(cross_mean)
            result[f"{c}_r{f}"] = peer_rank(c, peers)
            result[f"{c}_b{f}"] = rank_blend([f"{c}_p{f}", f"{c}_r{f}"])
            name = f"{c}_n{f}"
            result[name] = neutralize(f"{c}_b{f}", CHARS4)
            neutral_cols.append(name)
    neutral_ranks = result.groupby("date")[neutral_cols].rank(pct=True) - 0.5
    lower = neutral_ranks.quantile(0.25, axis=1)
    upper = neutral_ranks.quantile(0.75, axis=1)
    # IQR Winsor 均值：各列先截到本行四分位区间，再取均值（中间一半的票全部参与，而不只看两个分位点）。
    result["iqr"] = neutral_ranks.clip(lower=lower, upper=upper, axis=0).mean(axis=1, skipna=True)
    # 146：慢特征组合 = 低波动 / 小盘 / 低价三者截面排名等权相加，再与反转主信号按 1 : 0.5 做截面排名融合。
    char_ranks = result.groupby("date")[["c_park", "c_size", "c_lowpx"]].rank(pct=True)
    result["slow"] = char_ranks["c_park"] + char_ranks["c_size"] + char_ranks["c_lowpx"]
    result["factor"] = rank_blend(["iqr", "slow"], [1.0, 0.5])
    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan).astype("float64").fillna(0.0)
    return result[["date", "instrument", "factor"]].sort_values(
        ["date", "instrument"]
    ).reset_index(drop=True)


def main_b11(datasources, start_date, end_date):
    """【1397：临时峰度反向】新根单独版（公榜探针），STAT = A_kurt，方向 反向。
    根 = 1370_分布形态框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b11(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "A_kurt"))
    out = pd.concat(parts, ignore_index=True)
    out["factor"] = -out["factor"]                                           # 反向交付（09:30 仍为 0）
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b11(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z", "A"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(ABS(r)) OVER w AS cabs, SUM(r * r) OVER w AS crr, COUNT(r) OVER w AS cnr, SUM(r * r * r * r) OVER w AS cr4
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "A_kurt": lambda: df["cnr"] * df["cr4"] / (df["crr"] ** 2).where(df["crr"] > 0),
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b12(datasources, start_date, end_date):
    """【1397：临时五日偏度反向】新根单独版（公榜探针），STAT = A_skew5，方向 反向。
    根 = 1370_分布形态框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b12(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "A_skew5"))
    out = pd.concat(parts, ignore_index=True)
    out["factor"] = -out["factor"]                                           # 反向交付（09:30 仍为 0）
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b12(datasources, start_date, end_date, STAT):
    """【新根因子 第六轮 L：资金流窗口框架（1340 起）】一份代码覆盖 L/M/N/O/P/Q/R/S/T/U/V 十一个方向，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    分钟口径：1 分钟 bar 用结束时刻标注，截面 T 取标签恰为 T 的那条 bar（含）之前的连续竞价分钟，
    不含集合竞价 bar（09:30 及之前）；"当日至今"的量都按 (股票, 交易日) 分区、按时间累计，不跨日。
    30 分钟段口径：(09:30,10:00] … (14:30,15:00]，只用截面所在交易日之前的完整交易日（多日统计）
    或截面时刻及之前的段（T_acc）。不读 vwap 表、不读暴露表、不含自身收益反转写法。

    STAT：
      L_lab10/20/40  下一个半小时段在过去 N 日的"段收盘 ÷ 段 VWAP − 1"均值（同时段季节性）
      L_ret20        下一个半小时段在过去 20 日的段收益均值；L_lab20_dm 去掉各时段平均
      M_q / M_q_lin  聪明钱：以 |r|/√v（或 |r|/v）加权的当日 VWAP ÷ 当日 VWAP − 1
      N_dir / N_big  大单方向：Σ r·(ln 单笔额 − 之前分钟均值) ÷ Σ|r|；大单分钟收益和 − 小单分钟收益和
      Q_eff / Q_seff / Q_ac1  路径效率 |净变动|÷Σ|r|、带符号版、分钟收益一阶自相关
      U_nj / U_sj    当日跳跃次数（|r| > 4 倍之前分钟波动）与带符号跳跃收益和
      S_osz / S_osz30  买方每笔委托挂单量 ÷ 卖方（1–3 档，对数）当日均值 / 近 30 分钟均值
      O_q5           一档买量 ÷ 卖量（对数）近 5 分钟均值
      T_acc          本段成交额 ÷ 上一段（对数）× 本段收益符号
      R_round / R_half  当前价相对最近整数元 / 半元的带符号距离（÷ 价格）
      V_spr          当日相对价差均值
      W_vdir         量价同向：Σ r·(ln 成交量 − 之前分钟均值) ÷ Σ|r|
      W_cnt          1–3 档委托笔数不平衡 (买 − 卖)/(买 + 卖) 的当日均值
      W_hhi          当日分钟成交量集中度 Σv² ÷ (Σv)²
      W_zero         当日零收益分钟占比（价格黏滞 / 跳价约束）
      W_atsd         当日分钟平均单笔成交额（对数）的离散度
      W_hl           Σ(分钟最高 − 最低)/收盘 ÷ |当日净变动|（日内往返程度）
      X_q60 / X_q30  近 60 / 30 分钟带符号路径效率；X_q5d 近 5 日（1200 分钟）带符号路径效率
      X_n5d / X_m5d  近 5 日大单方向 / 聪明钱（|r|/v 加权 VWAP ÷ VWAP − 1）
      X_m75 / X_ma   聪明钱权重 |r|/v^0.75、|r|/成交额
      L_rest20       当日剩余各时段过去 20 日"收盘 ÷ VWAP − 1"均值的平均（季节性，换手更低）
      P_ac20 / P_ac60  个股过去 20/60 日相邻半小时段收益的一阶自相关（日内常数，供条件化头部用）
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    NEXT = {"1000": "1030", "1030": "1100", "1100": "1130", "1130": "1330", "1330": "1400", "1400": "1430", "1430": "1500"}
    bar1m = datasources["bar1m"]
    fam = STAT.split("_")[0]

    if fam in ("M", "N", "Q", "U", "S", "O", "R", "V", "W", "X", "Y", "Z", "A"):
        sql_min = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M') AS hm,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c, CAST(b.high AS DOUBLE) AS h, CAST(b.low AS DOUBLE) AS l,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CAST(COALESCE(b.deal_number, 0) AS DOUBLE) AS n,
                CAST(COALESCE(b.bid_volume1, 0) + COALESCE(b.bid_volume2, 0) + COALESCE(b.bid_volume3, 0) AS DOUBLE) AS bv,
                CAST(COALESCE(b.ask_volume1, 0) + COALESCE(b.ask_volume2, 0) + COALESCE(b.ask_volume3, 0) AS DOUBLE) AS av,
                CAST(COALESCE(b.bid_num_orders1, 0) + COALESCE(b.bid_num_orders2, 0) + COALESCE(b.bid_num_orders3, 0) AS DOUBLE) AS bo,
                CAST(COALESCE(b.ask_num_orders1, 0) + COALESCE(b.ask_num_orders2, 0) + COALESCE(b.ask_num_orders3, 0) AS DOUBLE) AS ao,
                CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.ask_volume1, 0) AS DOUBLE) AS av1,
                CAST(b.bid_price1 AS DOUBLE) AS bp1, CAST(b.ask_price1 AS DOUBLE) AS ap1
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        ), m1 AS (
            SELECT *, LN(c / COALESCE(LAG(c) OVER (PARTITION BY instrument_id, d ORDER BY date), o)) AS r,
                CASE WHEN n > 0 AND a > 0 THEN LN(a / n) END AS lats, CASE WHEN v > 0 THEN LN(v) END AS lv
            FROM b
        ), m2 AS (
            SELECT *
            FROM m1
        ), m3 AS (
            SELECT instrument_id, instrument, d, hm, date, c,
                SUM(a) OVER w AS ca, SUM(v) OVER w AS cv,
                SUM(ABS(r)) OVER w AS cabs,
                SUM(r * r * r) OVER w5 AS cr3_5, SUM(r * r) OVER w5 AS crr_5, COUNT(r) OVER w5 AS n5r
            FROM m2
            WINDOW w AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW),
                   w60 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW),
                   w30 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 29 PRECEDING AND CURRENT ROW),
                   w5 AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 1199 PRECEDING AND CURRENT ROW),
                   w90 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 89 PRECEDING AND CURRENT ROW),
                   w120 AS (PARTITION BY instrument_id, d ORDER BY date ROWS BETWEEN 119 PRECEDING AND CURRENT ROW),
                   w60x AS (PARTITION BY instrument_id ORDER BY date ROWS BETWEEN 59 PRECEDING AND CURRENT ROW)
        )
        SELECT * FROM m3 WHERE hm IN ('1000', '1030', '1100', '1130', '1330', '1400', '1430')
        """
        q5 = (pd.to_datetime(start_date) - pd.Timedelta(days=12)).strftime("%Y-%m-%d %H:%M:%S")     # 近 5 日窗口需要之前的分钟
        df = dai.query(sql_min, filters={"date": [q5, end_date]}, compression=True).df()
        df = df[pd.to_datetime(df["date"]) >= pd.Timestamp(start_date)].reset_index(drop=True)
        if not len(df):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        with np.errstate(all="ignore"):
            vw = df["ca"] / df["cv"].where(df["cv"] > 0)
            cabs = df["cabs"].where(df["cabs"] > 0)
            sig = {
                "A_skew5": lambda: np.sqrt(df["n5r"]) * df["cr3_5"] / (df["crr_5"] ** 1.5).where(df["crr_5"] > 0),
            }[STAT.split("_res")[0]]()
        df["signal"] = np.asarray(sig, dtype=float)
        df = df[["date", "instrument", "signal"]]
    else:
        q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
        sql_seg = f"""
        WITH b AS (
            SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
                CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
                CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
                CASE
                    WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                    WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                    WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                    WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                    WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                    WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                    WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                    WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
                END AS seg
            FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
            WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
              AND b.close > 0 AND b.open > 0
        )
        SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
            SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
        FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
        """
        sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
        if not len(sg):
            return pd.DataFrame(columns=["date", "instrument", "factor"])
        sg["instrument"] = sg["instrument"].astype(str)
        sg["d"] = pd.to_datetime(sg["d"])
        sg = sg.sort_values(["instrument", "d", "seg"]).reset_index(drop=True)
        with np.errstate(all="ignore"):
            sg["lab"] = sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1
            prev_c = sg.groupby(["instrument", "d"])["c_last"].shift(1)
            sg["ret"] = sg["c_last"] / prev_c.fillna(sg["o_first"]) - 1
        days = np.sort(sg["d"].unique())
        out_days = days[days >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
        rows = []
        if fam == "L":
            col = "ret" if STAT.startswith("L_ret") else "lab"
            N = int("".join(ch for ch in STAT.split("_")[1] if ch.isdigit()))
            piv = sg.pivot_table(index="d", columns=["instrument", "seg"], values=col, aggfunc="last").reindex(days)
            mu = piv.rolling(N, min_periods=max(N // 2, 3)).mean().shift(1)       # 只用之前的交易日
            if STAT.startswith("L_rest"):                                          # 当日剩余时段的平均季节性
                segs = ["1000", "1030", "1100", "1130", "1330", "1400", "1430", "1500"]
                for t in SLOTS:
                    rest = segs[segs.index(NEXT[t]):]
                    s = mu.loc[out_days].loc[:, mu.columns.get_level_values(1).isin(rest)].T.groupby(level=0).mean().T
                    s = s.stack().rename("signal").reset_index()
                    s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                    rows.append(s[["date", "instrument", "signal"]])
            if STAT.endswith("_dm"):
                mu = mu - mu.T.groupby(level=0).transform("mean").T
            for t in ([] if STAT.startswith("L_rest") else SLOTS):
                nx = NEXT[t]
                s = mu.loc[out_days].xs(nx, axis=1, level=1)
                s = s.stack().rename("signal").reset_index()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        elif fam == "P":
            N = int(STAT.split("ac")[1])
            sg["ret_next"] = sg.groupby(["instrument", "d"])["ret"].shift(-1)
            g = sg.assign(xy=sg["ret"] * sg["ret_next"], xx=sg["ret"] ** 2).groupby(["d", "instrument"])[["xy", "xx"]].sum()
            xy = g["xy"].unstack().reindex(days)
            xx = g["xx"].unstack().reindex(days)
            ac = (xy.rolling(N, min_periods=N // 2).sum() / xx.rolling(N, min_periods=N // 2).sum()).shift(1)
            s0 = ac.loc[out_days].stack().rename("signal").reset_index()
            for t in SLOTS:
                s = s0.copy()
                s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
                rows.append(s[["date", "instrument", "signal"]])
        else:                                                                      # T_acc
            prev_amt = sg.groupby("instrument")["amt"].shift(1)
            with np.errstate(all="ignore"):
                sg["signal"] = np.log(sg["amt"] / prev_amt.where(prev_amt > 0)) * np.sign(sg["ret"])
            s = sg[sg["seg"].isin(SLOTS) & sg["d"].isin(out_days)].copy()
            s["date"] = s["d"] + pd.to_timedelta(s["seg"].str[:2].astype(int), "h") + pd.to_timedelta(s["seg"].str[2:].astype(int), "m")
            rows.append(s[["date", "instrument", "signal"]])
        df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b13(datasources, start_date, end_date):
    """【1406：前日收盘竞价价格冲击】新根单独版（公榜探针），STAT = O_cauc，方向 正向。
    根 = 1400_集合竞价框架.py 原样复用；按自然月分批调用，控制平台整段调用时的内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_root_impl_b13(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), "O_cauc"))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _root_impl_b13(datasources, start_date, end_date, STAT):
    """【新根因子 第十轮：集合竞价与隔夜框架（1400 起）】一份代码覆盖开盘集合竞价、隔夜、前一日收盘集合竞价、首半小时，STAT 选一个统计量。
    同一段 SQL 被所有变体共用（评测机按 SQL 文本缓存），变体只改 STAT 一行。

    数据口径（1 分钟表）：09:16–09:24 的 bar 成交量为 0，收盘价是集合竞价的虚拟撮合价；一档买、卖价量相同（= 撮合价与可撮合量），
    未匹配量只出现在二档的一侧（价格记 0）；09:25 的 bar 是开盘集合竞价的成交（约 55% 的股票这一 bar 有成交量，其余为 0，
    竞价量类变体只覆盖前者）；15:00 的 bar 含收盘集合竞价成交，14:57 的 bar 收盘价是收盘竞价前的最后连续竞价价。
    当日只用 10:00 及之前的 bar（所有截面都在 10:00 及之后）；当日收盘、全天成交量等一律取前一交易日（按股票 shift(1)）。
    每个交易日一个值，7 个截面相同（日内常数，换手低）。不读 vwap 表、不读暴露表。

    STAT：
      O_gap     隔夜跳空：开盘竞价价 ÷ 前一日收盘 − 1
      O_aucv20  开盘竞价成交量 ÷ 过去 20 个交易日开盘竞价成交量均值（对数）
      O_aucs    开盘竞价成交量 ÷ 前一日全天成交量
      O_pre     竞价后段价格变动：开盘竞价价 ÷ 09:20 虚拟撮合价 − 1（09:20 之后不能撤单）
      O_imb     09:24 未匹配量不平衡：(未匹配买 − 未匹配卖) ÷ (可撮合量 + 未匹配买 + 未匹配卖)
      O_cauc    前一日收盘竞价的价格冲击：前一日收盘 ÷ 前一日 14:57 价 − 1
      O_caucs   前一日收盘竞价成交量占全天比例
      O_f30     首半小时收益：10:00 价 ÷ 开盘竞价价 − 1
      O_pre1    竞价前段（可撤单阶段）价格变动：09:20 虚拟撮合价 ÷ 09:16 虚拟撮合价 − 1
      O_lim1    前一日收盘涨停（1）／跌停（−1）／其他（0）；涨跌幅限制按代码：688、300、301 开头 20%，8、4 开头 30%，其余 10%（ST 无法识别，按 10%）
      O_lim5    过去 5 个交易日收盘涨停次数 − 跌停次数
    """
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    bar1m = datasources["bar1m"]
    q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=45)).strftime("%Y-%m-%d %H:%M:%S")   # 20 个交易日回看 + 前一日
    sql = f"""
    WITH b AS (
        SELECT b.instrument_id, ai.instrument, CAST(b.date AS DATE) AS d, strftime(b.date, '%H%M%S') AS t,
            CAST(b.close AS DOUBLE) AS c, CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v,
            CAST(COALESCE(b.bid_volume1, 0) AS DOUBLE) AS bv1, CAST(COALESCE(b.bid_volume2, 0) AS DOUBLE) AS bv2,
            CAST(COALESCE(b.ask_volume2, 0) AS DOUBLE) AS av2
        FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
        WHERE b.close > 0
    )
    SELECT instrument_id, MAX(instrument) AS instrument, d,
        MAX(CASE WHEN t = '091600' THEN c END) AS px0916, MAX(CASE WHEN t = '092000' THEN c END) AS px0920,
        MAX(CASE WHEN t = '092400' THEN bv1 END) AS m0924, MAX(CASE WHEN t = '092400' THEN bv2 END) AS ub0924,
        MAX(CASE WHEN t = '092400' THEN av2 END) AS ua0924,
        MAX(CASE WHEN t = '092500' THEN c END) AS apx, MAX(CASE WHEN t = '092500' THEN v END) AS aucv,
        MAX(CASE WHEN t = '100000' THEN c END) AS c1000,
        MAX(CASE WHEN t = '145700' THEN c END) AS c1457,
        MAX(CASE WHEN t = '150000' THEN c END) AS cls, MAX(CASE WHEN t = '150000' THEN v END) AS cauv,
        SUM(v) AS dv
    FROM b GROUP BY instrument_id, d
    """
    g = dai.query(sql, filters={"date": [q0, end_date]}, compression=True).df()
    if not len(g):
        return pd.DataFrame(columns=["date", "instrument", "factor"])
    g["instrument"] = g["instrument"].astype(str)
    g["d"] = pd.to_datetime(g["d"])
    g = g.sort_values(["instrument", "d"]).reset_index(drop=True)
    grp = g.groupby("instrument", sort=False)
    with np.errstate(all="ignore"):
        prev_cls = grp["cls"].shift(1)                                   # 只用前一交易日的收盘、全天量、收盘竞价
        prev_dv = grp["dv"].shift(1)
        prev_c1457 = grp["c1457"].shift(1)
        prev_cauv = grp["cauv"].shift(1)
        mu20 = grp["aucv"].transform(lambda s: s.shift(1).rolling(20, min_periods=10).mean())
        tot = g["m0924"] + g["ub0924"] + g["ua0924"]
        _code = g["instrument"].str[:3]
        _lim = np.where(_code.isin(["688", "300", "301"]), 0.20, np.where(g["instrument"].str[:1].isin(["8", "4"]), 0.30, 0.10))
        _pc = grp["cls"].shift(1)                                        # 涨跌停判定用"当日收盘 vs 前一日收盘"，再整体后移一天才用
        _up = (g["cls"] >= np.round(_pc * (1 + _lim), 2) - 1e-6).astype(float).where(_pc > 0)
        _dn = (g["cls"] <= np.round(_pc * (1 - _lim), 2) + 1e-6).astype(float).where(_pc > 0)
        g["_lstate"] = _up - _dn
        lim1 = g.groupby("instrument", sort=False)["_lstate"].shift(1)
        lim5 = g.groupby("instrument", sort=False)["_lstate"].transform(lambda s: s.shift(1).rolling(5, min_periods=3).sum())
        sig = {
            "O_gap": g["apx"] / prev_cls.where(prev_cls > 0) - 1,
            "O_aucv20": np.log(g["aucv"].where(g["aucv"] > 0) / mu20.where(mu20 > 0)),
            "O_aucs": g["aucv"] / prev_dv.where(prev_dv > 0),
            "O_pre": g["apx"] / g["px0920"].where(g["px0920"] > 0) - 1,
            "O_imb": (g["ub0924"] - g["ua0924"]) / tot.where(tot > 0),
            "O_cauc": prev_cls / prev_c1457.where(prev_c1457 > 0) - 1,
            "O_caucs": prev_cauv / prev_dv.where(prev_dv > 0),
            "O_f30": g["c1000"] / g["apx"].where(g["apx"] > 0) - 1,
            "O_pre1": g["px0920"] / g["px0916"].where(g["px0916"] > 0) - 1,
            "O_lim1": lim1,
            "O_lim5": lim5,
        }[STAT]
    g["signal"] = np.asarray(sig, dtype=float)
    g = g[g["d"] >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
    rows = []
    for t in SLOTS:
        s = g[["d", "instrument", "signal"]].copy()
        s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
        rows.append(s[["date", "instrument", "signal"]])
    df = pd.concat(rows, ignore_index=True)

    df["date"] = pd.to_datetime(df["date"])
    df["instrument"] = df["instrument"].astype(str)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def main_b14(datasources, start_date, end_date):
    """【收盘段季节性 L_close20】最后半小时 (14:30,15:00]（收盘集合竞价决定该段收盘价）的"段收盘 ÷ 段 VWAP − 1"在过去 20 个交易日的均值；
    同一交易日内各截面（10:00…14:30）取值相同，逐截面百分位排名 − 0.5，09:30 置 0。只用截面所在交易日之前的完整交易日。
    30 分钟段与 1232（同时段季节性）同一段 SQL：1 分钟 bar 聚成 (09:30,10:00] … (14:30,15:00]。按自然月分批调用，控制内存。
    """
    import pandas as pd
    parts = []
    for _ms in pd.period_range(pd.Timestamp(start_date).to_period("M"), pd.Timestamp(end_date).to_period("M"), freq="M"):
        _s = max(pd.Timestamp(start_date), _ms.start_time)
        _e = min(pd.Timestamp(end_date), _ms.end_time)
        if _s > _e:
            continue
        parts.append(_close_impl_b14(datasources, _s.strftime("%Y-%m-%d %H:%M:%S"), _e.strftime("%Y-%m-%d %H:%M:%S"), 20))
    out = pd.concat(parts, ignore_index=True)
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _close_impl_b14(datasources, start_date, end_date, N):
    import numpy as np
    import pandas as pd
    import dai

    SLOTS = ["1000", "1030", "1100", "1130", "1330", "1400", "1430"]
    bar1m = datasources["bar1m"]
    q0 = (pd.to_datetime(start_date) - pd.Timedelta(days=130)).strftime("%Y-%m-%d %H:%M:%S")
    sql_seg = f"""
    WITH b AS (
        SELECT b.instrument_id, ai.instrument, b.date, CAST(b.date AS DATE) AS d,
            CAST(b.open AS DOUBLE) AS o, CAST(b.close AS DOUBLE) AS c,
            CAST(COALESCE(b.volume, 0) AS DOUBLE) AS v, CAST(COALESCE(b.amount, 0) AS DOUBLE) AS a,
            CASE
                WHEN strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '100000' THEN '1000'
                WHEN strftime(b.date, '%H%M%S') > '100000' AND strftime(b.date, '%H%M%S') <= '103000' THEN '1030'
                WHEN strftime(b.date, '%H%M%S') > '103000' AND strftime(b.date, '%H%M%S') <= '110000' THEN '1100'
                WHEN strftime(b.date, '%H%M%S') > '110000' AND strftime(b.date, '%H%M%S') <= '113000' THEN '1130'
                WHEN strftime(b.date, '%H%M%S') > '130000' AND strftime(b.date, '%H%M%S') <= '133000' THEN '1330'
                WHEN strftime(b.date, '%H%M%S') > '133000' AND strftime(b.date, '%H%M%S') <= '140000' THEN '1400'
                WHEN strftime(b.date, '%H%M%S') > '140000' AND strftime(b.date, '%H%M%S') <= '143000' THEN '1430'
                WHEN strftime(b.date, '%H%M%S') > '143000' AND strftime(b.date, '%H%M%S') <= '150000' THEN '1500'
            END AS seg
        FROM {bar1m} b LEFT JOIN all_instruments ai USING (instrument_id)
        WHERE strftime(b.date, '%H%M%S') > '093000' AND strftime(b.date, '%H%M%S') <= '150000'
          AND b.close > 0 AND b.open > 0
    )
    SELECT MAX(date) AS date, instrument_id, MAX(instrument) AS instrument, d, seg,
        SUM(a) AS amt, SUM(v) AS vol, arg_max(c, date) AS c_last, arg_min(o, date) AS o_first
    FROM b WHERE seg IS NOT NULL GROUP BY instrument_id, d, seg
    """
    sg = dai.query(sql_seg, filters={"date": [q0, end_date]}, compression=True).df()
    if not len(sg):
        return pd.DataFrame(columns=["date", "instrument", "factor"])
    sg["instrument"] = sg["instrument"].astype(str)
    sg["d"] = pd.to_datetime(sg["d"])
    sg = sg[sg["seg"] == "1500"]
    with np.errstate(all="ignore"):
        sg = sg.assign(lab=sg["c_last"] / (sg["amt"] / sg["vol"].where(sg["vol"] > 0)) - 1)
    piv = sg.pivot_table(index="d", columns="instrument", values="lab", aggfunc="last").sort_index()
    mu = piv.rolling(N, min_periods=N // 2).mean().shift(1)                  # 只用之前的交易日
    out_days = mu.index[mu.index >= pd.Timestamp(pd.to_datetime(start_date).normalize())]
    s0 = mu.loc[out_days].stack().rename("signal").reset_index()
    rows = []
    for t in SLOTS:
        s = s0.copy()
        s["date"] = s["d"] + pd.Timedelta(hours=int(t[:2]), minutes=int(t[2:]))
        rows.append(s[["date", "instrument", "signal"]])
    df = pd.concat(rows, ignore_index=True)
    df["signal"] = df["signal"].replace([np.inf, -np.inf], np.nan)
    df = df.dropna(subset=["instrument", "signal"]).drop_duplicates(["date", "instrument"])
    df["factor"] = df.groupby("date")["signal"].rank(pct=True) - 0.5
    _slots = {"0930", "1000", "1030", "1100", "1130", "1330", "1400", "1430"}
    _inst_tbl = datasources.get("instruments", "cpt_jyc_2026_instruments")
    _pool = dai.query(f"SELECT date, instrument FROM {_inst_tbl}",
                      filters={"date": [start_date, end_date]}, compression=True).df()
    _pool["date"] = pd.to_datetime(_pool["date"])
    _pool["instrument"] = _pool["instrument"].astype(str)
    _pool = _pool[_pool["date"].dt.strftime("%H%M").isin(_slots)].drop_duplicates(["date", "instrument"])
    out = _pool.merge(df[["date", "instrument", "factor"]], on=["date", "instrument"], how="left")
    out["factor"] = out["factor"].astype("float64").fillna(0.0)
    out.loc[out["date"].dt.strftime("%H%M") == "0930", "factor"] = 0.0
    return out[["date", "instrument", "factor"]].sort_values(["date", "instrument"]).reset_index(drop=True)


def _w5_safe(df):
    """近 5 日（1200 分钟）窗口的平台安全写法（10-19，复活 Y_sd5／X_n5d）。
    SQL 只给按 (股票, 交易日) 分区的前缀和，并多取 15:00 一行作全天合计。这里拼成
      窗口(k, t) = 前缀(k, t) + 累计(k−1) − 累计(k−6) − 前缀(k−5, t)，
    k 为该股票的第 k 个交易日。每天 240 行（09:31–15:00），无缺分钟时与原 ROWS BETWEEN 1199 PRECEDING 写法相同。
    产出 a5d、nd5（由 cabs、cnd）与 sd5（由 sl、sl2、nl），最后去掉 15:00 行。"""
    import numpy as np
    import pandas as pd
    cols = [c for c in ("cabs", "cnd", "sl", "sl2", "nl") if c in df.columns]
    df = df.copy()
    df["hm"] = df["hm"].astype(str)
    P = df[cols].apply(pd.to_numeric, errors="coerce").astype("float64").fillna(0.0)
    tot = df.loc[df["hm"] == "1500", ["instrument_id", "d"]].copy()
    tot[cols] = P.loc[tot.index].to_numpy()
    tot = tot.drop_duplicates(["instrument_id", "d"]).sort_values(["instrument_id", "d"]).reset_index(drop=True)
    tot["k"] = tot.groupby("instrument_id").cumcount()
    C = tot.groupby("instrument_id")[cols].cumsum().to_numpy()
    ck = pd.DataFrame(C, columns=cols, index=pd.MultiIndex.from_arrays([tot["instrument_id"].to_numpy(), tot["k"].to_numpy()]))
    x = df[["instrument_id", "d"]].merge(tot[["instrument_id", "d", "k"]], on=["instrument_id", "d"], how="left")
    kk = x["k"].to_numpy(dtype="float64")
    ki = np.where(np.isfinite(kk), kk, -999).astype(np.int64)
    iid = df["instrument_id"].to_numpy()

    def at(shift):
        return ck.reindex(pd.MultiIndex.from_arrays([iid, ki - shift])).to_numpy()

    c1, c6 = at(1), at(6)
    c6 = np.where((ki == 5)[:, None], 0.0, c6)
    pk = P.copy()
    pk["instrument_id"], pk["k"], pk["hm"] = iid, ki, df["hm"].to_numpy()
    pk = pk[pk["k"] >= 0].drop_duplicates(["instrument_id", "k", "hm"]).set_index(["instrument_id", "k", "hm"])[cols]
    p5 = np.nan_to_num(pk.reindex(pd.MultiIndex.from_arrays([iid, ki - 5, df["hm"].to_numpy()])).to_numpy(), nan=0.0)
    Wv = P.to_numpy() + c1 - c6 - p5
    Wv[ki < 5] = np.nan
    w = pd.DataFrame(Wv, columns=cols, index=df.index)
    if "cabs" in cols:
        df["a5d"] = w["cabs"]
    if "cnd" in cols:
        df["nd5"] = w["cnd"]
    if "sl" in cols:
        n_, s1, s2 = w["nl"], w["sl"], w["sl2"]
        var = (s2 - s1 * s1 / n_) / (n_ - 1)
        df["sd5"] = np.sqrt(var.where((n_ > 1) & (var >= 0)))
    return df[df["hm"] != "1500"].reset_index(drop=True)
